In [1]:
# 匹配s2的raw文件和plume id
import os
import re
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta, timezone

import rasterio
from rasterio.warp import transform as rio_transform

RAW_SAFE_DIR = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/data_download/raw_data_dir_s2"
PLUME_CSV    = "/data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file.csv"
OUT_CSV      = "./plume_to_safe_recovered_bucketed_2.csv"

# 你的逻辑是 24h 内找第一个：±1天通常够；如果你担心 UTC 日界/跨天，改成 2
WINDOW_DAYS = 1

SAFE_RE = re.compile(
    r"^(S2[ABC]_MSIL2A)_(\d{8}T\d{6})_N\d{4}_R\d{3}_(T\d{2}[A-Z]{3})_(\d{8}T\d{6})\.SAFE$"
)

def parse_plume_time(tstr: str) -> datetime:
    dt = datetime.fromisoformat(tstr)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)

def safe_sensing_time(safe_name: str):
    m = SAFE_RE.match(safe_name)
    if not m:
        return None
    return datetime.strptime(m.group(2), "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)

def build_buckets(raw_dir: str):
    """
    Return: buckets[YYYY-MM-DD] -> list[(sensing_time, safe_name, safe_dir)] sorted by sensing_time
    """
    buckets = {}
    total = 0
    matched = 0

    for name in os.listdir(raw_dir):
        total += 1
        if not name.endswith(".SAFE"):
            continue
        st = safe_sensing_time(name)
        if st is None:
            continue
        matched += 1
        d = st.date().isoformat()
        buckets.setdefault(d, []).append((st, name, os.path.join(raw_dir, name)))

    for d in buckets:
        buckets[d].sort(key=lambda x: x[0])

    print(f"[bucket] scanned entries={total}, SAFE matched pattern={matched}, buckets={len(buckets)}")
    return buckets

def find_any_20m_jp2(safe_dir: str):
    p = Path(safe_dir)
    for pat in ["*_B11_20m.jp2", "*_B12_20m.jp2", "*_B8A_20m.jp2", "*_B02_20m.jp2", "*_20m.jp2"]:
        hits = list(p.glob(pat))
        if hits:
            return str(hits[0])
    return None

def safe_covers_point(jp2_path: str, lat: float, lon: float) -> bool:
    """
    Correct coverage test for JP2 (usually UTM CRS):
    EPSG:4326 lon/lat -> ds.crs x/y -> inverse affine -> pixel col/row
    """
    with rasterio.open(jp2_path) as ds:
        if ds.crs is None:
            return False

        # 1) WGS84 -> dataset CRS
        try:
            xs, ys = rio_transform("EPSG:4326", ds.crs, [lon], [lat])
            x, y = xs[0], ys[0]
        except Exception:
            return False

        # 2) CRS -> pixel (col,row)
        try:
            col, row = ~ds.transform * (x, y)
        except Exception:
            return False

        # 3) in bounds
        return (0 <= row < ds.height) and (0 <= col < ds.width)

def candidate_days(plume_dt: datetime, window_days: int):
    return [(plume_dt.date() + timedelta(days=k)).isoformat() for k in range(-window_days, window_days + 1)]

def recover_one(plume_dt: datetime, lat: float, lon: float, buckets):
    for d in candidate_days(plume_dt, WINDOW_DAYS):
        if d not in buckets:
            continue
        for st, safe_name, safe_dir in buckets[d]:
            # 严格 24h 窗口（和你当初下载逻辑一致）
            if abs((st - plume_dt).total_seconds()) > 24 * 3600:
                continue

            jp2 = find_any_20m_jp2(safe_dir)
            if jp2 is None:
                continue

            if safe_covers_point(jp2, lat, lon):
                return safe_name, jp2, st.isoformat()

    return None, None, None

def main():
    buckets = build_buckets(RAW_SAFE_DIR)

    df = pd.read_csv(PLUME_CSV)
    need = ["plume_id", "plume_latitude", "plume_longitude", "datetime"]
    for c in need:
        if c not in df.columns:
            raise RuntimeError(f"CSV missing column: {c}")

    out_rows = []
    hit = 0

    for i, r in df.iterrows():
        pid = str(r["plume_id"])
        lat = float(r["plume_latitude"])
        lon = float(r["plume_longitude"])
        dt = parse_plume_time(str(r["datetime"]))

        safe_name, jp2_path, sensing_iso = recover_one(dt, lat, lon, buckets)
        if safe_name is not None:
            hit += 1

        out_rows.append({
            "plume_id": pid,
            "datetime": dt.isoformat(),
            "lat": lat,
            "lon": lon,
            "matched_safe": safe_name,
            "matched_sensing_time": sensing_iso,
            "matched_jp2": jp2_path,
        })

        if (i + 1) % 200 == 0:
            print(f"processed {i+1}/{len(df)} hit={hit}")

    out = pd.DataFrame(out_rows)
    out.to_csv(OUT_CSV, index=False)
    print("Saved:", OUT_CSV)
    print(f"Matched {hit}/{len(out)} ({hit/len(out):.2%})")

if __name__ == "__main__":
    main()


[bucket] scanned entries=5753, SAFE matched pattern=4680, buckets=1104
processed 200/24470 hit=142
processed 400/24470 hit=278
processed 600/24470 hit=365
processed 800/24470 hit=431
processed 1000/24470 hit=556
processed 1200/24470 hit=662
processed 1400/24470 hit=759
processed 1600/24470 hit=893
processed 1800/24470 hit=1006
processed 2000/24470 hit=1145
processed 2200/24470 hit=1249
processed 2400/24470 hit=1430
processed 2600/24470 hit=1449
processed 2800/24470 hit=1451
processed 3000/24470 hit=1567
processed 3200/24470 hit=1713
processed 3400/24470 hit=1815
processed 3600/24470 hit=1882
processed 3800/24470 hit=1889
processed 4000/24470 hit=1939
processed 4200/24470 hit=2016
processed 4400/24470 hit=2096
processed 4600/24470 hit=2205
processed 4800/24470 hit=2282
processed 5000/24470 hit=2414
processed 5200/24470 hit=2532
processed 5400/24470 hit=2654
processed 5600/24470 hit=2840
processed 5800/24470 hit=3040
processed 6000/24470 hit=3178
processed 6200/24470 hit=3227
processed 6

In [ ]:
# 下载-7
import os
import time
import json
import random
import shutil
import zipfile
import threading
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

CDSE_USER='yuyao16@ualberta.ca' 
CDSE_PASS='finhah-3zihty-seHmuf'

# =========================
# Config
# =========================
MANIFEST_CSV = "/data2/yuyao/methane_emission/preprocess_dataset_s2/manifest_minus7_plume_to_safe.csv"
MANIFEST_OUT = "/data2/yuyao/methane_emission/preprocess_dataset_s2/manifest_minus7_plume_to_safe.csv"  # 原地覆盖即可

RAW_DIR = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/data_download/raw_data_dir_s2_-7"

# CDSE credentials from env
# CDSE_USER = os.environ.get("CDSE_USER", "")
# CDSE_PASS = os.environ.get("CDSE_PASS", "")
if not CDSE_USER or not CDSE_PASS:
    print("[WARN] CDSE_USER / CDSE_PASS not set. Please:")
    print("  export CDSE_USER='xxx' ; export CDSE_PASS='yyy'")

# Download throttling
ZIPPER_MAX_CONCURRENT = 1          # 强烈建议 1，避免 429
ZIPPER_MIN_INTERVAL_SEC = 1.2      # 每次 zipper 请求最小间隔
MAX_WORKERS = 4                    # 并行度（product 级别）；ZIPPER_MAX_CONCURRENT 会限制真正下载并发

# Retry
MAX_TRIES = 10
BASE_SLEEP = 2.0

# Extract: only keep R20m jp2 + MTD xml
KEEP_R20M_ONLY = True

# Progress print interval
PRINT_EVERY_PRODUCTS = 1

# =========================
# Utils
# =========================
def debug(msg: str) -> None:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}][pid:{os.getpid()}][tid:{threading.get_ident()}] {msg}", flush=True)

def safe_mkdir(p: str) -> None:
    os.makedirs(p, exist_ok=True)

def raw_exists(product_name: str) -> bool:
    if not isinstance(product_name, str) or not product_name:
        return False
    # 你的目录里是 .../<product_name>.SAFE （文件夹）以及可能有 zip
    d = os.path.join(RAW_DIR, product_name)
    z1 = os.path.join(RAW_DIR, product_name + ".zip")
    z2 = os.path.join(RAW_DIR, product_name + ".SAFE.zip")
    return os.path.isdir(d) or os.path.exists(z1) or os.path.exists(z2)

def extract_marker_path(product_name: str) -> str:
    return os.path.join(RAW_DIR, product_name, ".extract_complete")

# =========================
# Rate limiter & locks
# =========================
_zipper_sema = threading.Semaphore(ZIPPER_MAX_CONCURRENT)

class RateLimiter:
    def __init__(self, min_interval_sec: float):
        self.min_interval = float(min_interval_sec)
        self._lock = threading.Lock()
        self._next_time = 0.0

    def wait(self):
        with self._lock:
            now = time.time()
            if now < self._next_time:
                time.sleep(self._next_time - now)
            self._next_time = time.time() + self.min_interval

zipper_rl = RateLimiter(ZIPPER_MIN_INTERVAL_SEC)

# per-product lock (avoid double download/extract of same product)
_product_locks = {}
_product_locks_lock = threading.Lock()

def get_product_lock(name: str) -> threading.Lock:
    with _product_locks_lock:
        if name not in _product_locks:
            _product_locks[name] = threading.Lock()
        return _product_locks[name]

# =========================
# CDSE Auth
# =========================
def get_access_token(username: str, password: str) -> str:
    data = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password",
    }
    r = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data=data,
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["access_token"]

class RefreshableAccessToken:
    def __init__(self, user, pwd) -> None:
        self.user = user
        self.pwd = pwd
        self.value = get_access_token(user, pwd)
        self.lock = threading.Lock()

    def update(self):
        with self.lock:
            self.value = get_access_token(self.user, self.pwd)

    def get(self):
        with self.lock:
            return self.value

def refresh_token_daemon(token_obj: RefreshableAccessToken):
    while True:
        try:
            debug("refresh token")
            token_obj.update()
        except Exception as e:
            debug(f"token refresh failed: {e}")
        time.sleep(300)

# =========================
# HTTP request with retry
# =========================
def request_with_retry(method, url, session: requests.Session, *,
                       headers=None,
                       max_tries=MAX_TRIES,
                       base_sleep=BASE_SLEEP,
                       timeout=300,
                       stream=False):
    last_exc = None
    for attempt in range(1, max_tries + 1):
        try:
            resp = session.request(method, url, headers=headers, timeout=timeout, stream=stream)

            if resp.status_code == 429:
                ra = resp.headers.get("Retry-After")
                if ra is not None:
                    try:
                        sleep_s = float(ra)
                    except Exception:
                        sleep_s = base_sleep
                else:
                    sleep_s = base_sleep * (2 ** (attempt - 1))
                sleep_s += random.uniform(0, 0.6)
                debug(f"429 Too Many Requests -> sleep {sleep_s:.2f}s (attempt {attempt}/{max_tries})")
                time.sleep(sleep_s)
                continue

            if resp.status_code in (500, 502, 503, 504):
                sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0, 0.6)
                debug(f"{resp.status_code} server error -> sleep {sleep_s:.2f}s (attempt {attempt}/{max_tries})")
                time.sleep(sleep_s)
                continue

            resp.raise_for_status()
            return resp

        except requests.RequestException as e:
            last_exc = e
            sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0, 0.6)
            debug(f"request error: {e} -> sleep {sleep_s:.2f}s (attempt {attempt}/{max_tries})")
            time.sleep(sleep_s)

    raise last_exc if last_exc else RuntimeError("request_with_retry failed")

# =========================
# Download & Extract
# =========================
def download_product_zip(token: str, product_id: str, out_zip: str):
    """
    Download CDSE zipper product to out_zip.
    Skips if out_zip already exists and non-empty.
    """
    if os.path.exists(out_zip) and os.path.getsize(out_zip) > 0:
        return

    tmp = out_zip + ".part"
    if os.path.exists(tmp):
        try:
            os.remove(tmp)
        except Exception:
            pass

    url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"
    headers = {"Authorization": f"Bearer {token}"}

    with _zipper_sema:
        zipper_rl.wait()
        with requests.Session() as s:
            with request_with_retry("GET", url, s, headers=headers, stream=True, timeout=300) as r:
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp, out_zip)

def extract_r20m_from_zip(zip_path: str, out_dir: str):
    """
    Extract only R20m jp2 + MTD_MSIL2A.xml to out_dir
    and write marker .extract_complete when done.
    """
    safe_mkdir(out_dir)
    marker = os.path.join(out_dir, ".extract_complete")
    if os.path.exists(marker):
        return

    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        picks = []
        for n in names:
            nn = n.lower()
            if KEEP_R20M_ONLY:
                if ("granule/" in nn) and ("/img_data/r20m/" in nn) and nn.endswith(".jp2"):
                    picks.append(n)
                elif n.endswith("MTD_MSIL2A.xml"):
                    picks.append(n)
            else:
                # (not used) extract all
                picks.append(n)

        for n in picks:
            if n.endswith("/"):
                continue
            fn = os.path.basename(n)
            if not fn:
                continue
            target = os.path.join(out_dir, fn)
            if os.path.exists(target) and os.path.getsize(target) > 0:
                continue
            with z.open(n) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)

    with open(marker, "w") as f:
        f.write("ok")

# =========================
# Progress
# =========================
class ProdProgress:
    def __init__(self, total_products: int):
        self.total = total_products
        self.lock = threading.Lock()
        self.start = time.time()
        self.done = 0
        self.ok = 0
        self.skipped = 0
        self.fail = 0

    def update(self, status: str):
        with self.lock:
            self.done += 1
            if status == "ok":
                self.ok += 1
            elif status == "skipped":
                self.skipped += 1
            else:
                self.fail += 1

            if (self.done % PRINT_EVERY_PRODUCTS) == 0 or self.done == self.total:
                elapsed = time.time() - self.start
                rate = self.done / elapsed if elapsed > 0 else 0.0
                remain = (self.total - self.done) / rate if rate > 0 else -1
                debug(
                    f"[PRODUCT PROGRESS] {self.done}/{self.total} "
                    f"(ok={self.ok}, skipped={self.skipped}, fail={self.fail}) "
                    f"| elapsed={elapsed/3600:.2f}h | rate={rate:.3f} prod/s | ETA={remain/3600:.2f}h"
                )

# =========================
# Core: download one product
# =========================
def download_one_product(prod_name: str, prod_id: str, token_obj: RefreshableAccessToken):
    """
    Download and extract a product (once).
    Returns status: ok / skipped / fail
    """
    if not prod_name or not prod_id:
        return "fail", "missing prod_name/prod_id"

    prod_dir = os.path.join(RAW_DIR, prod_name)
    marker = os.path.join(prod_dir, ".extract_complete")

    # We will store zip as <prod_name>.zip
    zip_path = os.path.join(RAW_DIR, prod_name + ".zip")

    # lock per product
    lock = get_product_lock(prod_name)
    with lock:
        # if already extracted, skip
        if os.path.exists(marker):
            return "skipped", "already extracted"

        # if directory exists and seems to have jp2, also consider done (but still write marker)
        if os.path.isdir(prod_dir):
            jp2s = list(Path(prod_dir).glob("*.jp2"))
            if len(jp2s) > 0:
                with open(marker, "w") as f:
                    f.write("ok")
                return "skipped", "dir had jp2; marker written"

        # ensure RAW_DIR exists
        safe_mkdir(RAW_DIR)

        try:
            debug(f"[{prod_name}] downloading zip...")
            download_product_zip(token_obj.get(), prod_id, zip_path)

            debug(f"[{prod_name}] extracting R20m...")
            extract_r20m_from_zip(zip_path, prod_dir)

            return "ok", "download+extract ok"

        except Exception as e:
            return "fail", str(e)

# =========================
# Main
# =========================
if __name__ == "__main__":
    safe_mkdir(RAW_DIR)

    debug(f"load manifest: {MANIFEST_CSV}")
    df = pd.read_csv(MANIFEST_CSV)

    # sanity checks
    need_cols = ["ok", "raw_dir_exists", "s2_minus7_product_name", "s2_minus7_id"]
    for c in need_cols:
        if c not in df.columns:
            raise RuntimeError(f"manifest missing column: {c}")

    # target rows = ok==1 & raw_dir_exists==0
    tgt = df[(df["ok"] == 1) & (df["raw_dir_exists"] == 0)].copy()
    debug(f"rows needing raw download (case-level) = {len(tgt)}")

    # dedupe by product
    prod_df = tgt[["s2_minus7_product_name", "s2_minus7_id"]].dropna().drop_duplicates("s2_minus7_product_name")
    prod_df["s2_minus7_product_name"] = prod_df["s2_minus7_product_name"].astype(str)
    prod_df["s2_minus7_id"] = prod_df["s2_minus7_id"].astype(str)

    # filter out products that already exist on disk (double safety)
    products = []
    for _, r in prod_df.iterrows():
        name = r["s2_minus7_product_name"]
        pid = r["s2_minus7_id"]
        if raw_exists(name) or os.path.exists(extract_marker_path(name)):
            continue
        products.append((name, pid))

    debug(f"unique products to download = {len(products)} (after disk-skip)")

    if len(products) == 0:
        debug("nothing to download. exit.")
        raise SystemExit(0)

    # auth
    token_obj = RefreshableAccessToken(CDSE_USER, CDSE_PASS)
    th = threading.Thread(target=refresh_token_daemon, args=(token_obj,), daemon=True)
    th.start()

    prog = ProdProgress(total_products=len(products))

    # we will update df in a thread-safe way and flush to csv periodically
    df_lock = threading.Lock()
    flush_state = {"last": time.time()}
    FLUSH_EVERY_SEC = 01  # 每分钟写一次，避免频繁 IO TODO

    def mark_product_done(prod_name: str):
        with df_lock:
            # Set raw_dir_exists=1 for all rows with this product
            df.loc[df["s2_minus7_product_name"] == prod_name, "raw_dir_exists"] = 1

            # periodic flush
            now = time.time()
            if now - flush_state["last"] >= FLUSH_EVERY_SEC:
                tmp = MANIFEST_OUT + ".part"
                df.to_csv(tmp, index=False)
                os.replace(tmp, MANIFEST_OUT)
                flush_state["last"] = now
                debug(f"flushed manifest: {MANIFEST_OUT}")

    # run pool
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {}
        for name, pid in products:
            fut = ex.submit(download_one_product, name, pid, token_obj)
            futs[fut] = name

        for fut in as_completed(futs):
            name = futs[fut]
            try:
                status, info = fut.result()
            except Exception as e:
                status, info = "fail", f"exception: {e}"

            if status in ("ok", "skipped"):
                # ensure marker / raw exists -> then mark df
                if raw_exists(name) or os.path.exists(extract_marker_path(name)):
                    mark_product_done(name)
                prog.update(status="ok" if status == "ok" else "skipped")
                debug(f"[{name}] {status}: {info}")
            else:
                prog.update(status="fail")
                debug(f"[{name}] FAIL: {info}")

    # final flush
    with df_lock:
        tmp = MANIFEST_OUT + ".part"
        df.to_csv(tmp, index=False)
        os.replace(tmp, MANIFEST_OUT)
    debug(f"final saved manifest: {MANIFEST_OUT}")
    debug("all done")

[2026-01-15 05:33:34][pid:3982510][tid:140457180038208] load manifest: /data2/yuyao/methane_emission/preprocess_dataset_s2/manifest_minus7_plume_to_safe.csv
[2026-01-15 05:33:34][pid:3982510][tid:140457180038208] rows needing raw download (case-level) = 619
[2026-01-15 05:33:34][pid:3982510][tid:140457180038208] unique products to download = 227 (after disk-skip)
[2026-01-15 05:33:36][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 05:33:36][pid:3982510][tid:140452566214208] [S2C_MSIL2A_20250214T033901_N0511_R061_T48RVP_20250214T090512.SAFE] downloading zip...[2026-01-15 05:33:36][pid:3982510][tid:140452334396992] [S2B_MSIL2A_20250214T144739_N0511_R139_T20PMS_20250214T164302.SAFE] downloading zip...

[2026-01-15 05:33:36][pid:3982510][tid:140452557821504] [S2C_MSIL2A_20250209T175531_N0511_R141_T13TEE_20250209T210409.SAFE] downloading zip...
[2026-01-15 05:33:36][pid:3982510][tid:140452342789696] [S2C_MSIL2A_20250214T070021_N0511_R063_T42VWP_20250214T110112.SAFE] downloading

[2026-01-15 15:56:11][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:01:12][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:06:14][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:11:15][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:16:17][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:21:18][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:26:19][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:31:20][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:36:21][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:41:22][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:46:24][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:51:25][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 16:56:26][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 17:01:27][pid:3982510][tid:140452574606912] refresh token
[2026-01-15 17:06:30

In [1]:
# s2_minus7_shifted_georef_raw_to_512.py crop to 512有问题
import os
import re
import threading
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_origin

# =========================
# Config (EDIT THESE)
# =========================
MANIFEST_CSV = "/data2/yuyao/methane_emission/preprocess_dataset_s2/manifest_minus7_plume_to_safe.csv"
MERGED_CSV   = "/data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file.csv"

RAW_DIR      = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/data_download/raw_data_dir_s2_-7"
OUT_ROOT     = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_-790360_512"

WINDOW_SIZE  = 512
MAX_WORKERS  = 8

# plume bounds: lon/lat +/- 0.01 (same as your t0 download code)
DELTA_DEG = 0.01

# band jp2 matching (R20m)
JP2_PATTERN = re.compile(r".*B[0-9A-Za-z]+_20m\.jp2$", re.IGNORECASE)
TYPE_PATTERN = re.compile(r".*B([0-9A-Za-z]+)_20m\.jp2$", re.IGNORECASE)

# =========================
# Logging
# =========================
def debug(msg: str) -> None:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}][pid:{os.getpid()}][tid:{threading.get_ident()}] {msg}", flush=True)

# =========================
# Geo helpers (same idea as your t0 download code)
# =========================
def latlon_to_pixel(lat, lon, dataset):
    # dataset.crs is projected; rasterio handles transform
    from pyproj import Transformer
    transformer = Transformer.from_crs("EPSG:4326", dataset.crs, always_xy=True)
    x, y = transformer.transform(lon, lat)
    col, row = ~dataset.transform * (x, y)
    return col, row

def parse_a_file_window(file_path, plume_bounds):
    # plume_bounds = [lon_min, lat_min, lon_max, lat_max]
    with rasterio.open(file_path) as ds:
        # top-left = (lat_max, lon_min), bottom-right = (lat_min, lon_max)
        tl_col, tl_row = latlon_to_pixel(plume_bounds[3], plume_bounds[0], ds)
        br_col, br_row = latlon_to_pixel(plume_bounds[1], plume_bounds[2], ds)

        center_x = (tl_col + br_col) / 2.0
        center_y = (tl_row + br_row) / 2.0

        win = Window(center_x - WINDOW_SIZE // 2, center_y - WINDOW_SIZE // 2, WINDOW_SIZE, WINDOW_SIZE)
        arr = ds.read(1, window=win)  # (H,W)
        return arr

# =========================
# Write "GDAL true multiband" GeoTIFF (same writer idea as your std512 script)
# =========================
def write_gdal_multiband_tif(out_path: str, bhw: np.ndarray):
    bhw = np.asarray(bhw)
    if bhw.ndim != 3:
        raise ValueError(f"expect BHW, got {bhw.shape}")

    B, H, W = bhw.shape
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    transform = from_origin(0, 0, 1, 1)  # dummy
    profile = {
        "driver": "GTiff",
        "height": H,
        "width": W,
        "count": B,
        "dtype": str(bhw.dtype),
        "transform": transform,
        "compress": "deflate",
        "predictor": 2 if np.issubdtype(bhw.dtype, np.floating) else 1,
        "tiled": True,
        "blockxsize": 256 if W >= 256 else W,
        "blockysize": 256 if H >= 256 else H,
        "BIGTIFF": "IF_SAFER",
    }
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(bhw)

# =========================
# Build (12,512,512) from extracted jp2 folder
# =========================
def build_multiband_512_from_product(product_dir: str, plume_bounds, bug):
    jp2_dir = Path(product_dir)
    if (not jp2_dir.exists()) or (not jp2_dir.is_dir()):
        bug.append(f"raw_dir_missing:{product_dir}")
        return None

    jp2_files = []
    for p in jp2_dir.rglob("*.jp2"):
        if p.is_file() and JP2_PATTERN.match(str(p)):
            jp2_files.append(str(p))
    if not jp2_files:
        bug.append(f"no_jp2_found:{product_dir}")
        return None

    img = np.zeros((12, WINDOW_SIZE, WINDOW_SIZE), dtype=np.float32)

    seen = set()
    for fp in jp2_files:
        m = TYPE_PATTERN.match(os.path.basename(fp))
        if not m:
            continue
        bstr = m.group(1).upper()
        band = 8 if bstr == "8A" else int(bstr)
        if not (1 <= band <= 12):
            continue
        try:
            patch = parse_a_file_window(fp, plume_bounds)  # (512,512)
            if patch.shape != (WINDOW_SIZE, WINDOW_SIZE):
                bug.append(f"band{bstr}_not512:{patch.shape}")
                continue
            img[band - 1] = patch.astype(np.float32)
            seen.add(band)
        except Exception as e:
            bug.append(f"band{bstr}_read_err:{type(e).__name__}:{e}")

    expected = {1,2,3,4,5,6,7,8,11,12}  # 注意：这里的 8 表示 B8A（你原来的 band=8 if bstr=="8A"）
    # 解释：B08 和 B10 不在 expected 里 -> 默认用 0 填充，不算缺失
    missing = sorted(list(expected - seen))
    if missing:
        bug.append(f"bands_missing:{len(seen)}/{len(expected)} missing={missing}")
        # 仍然返回，让你能看统计/定位问题
    return img

# =========================
# One-row worker
# =========================
def process_one(plume_id: str, product_name: str, lat: float, lon: float):
    bug = []

    if not plume_id:
        return plume_id, False, "missing_plume_id"
    if not product_name:
        return plume_id, False, "missing_product_name"

    product_dir = os.path.join(RAW_DIR, product_name)
    out_dir = os.path.join(OUT_ROOT, plume_id)
    os.makedirs(out_dir, exist_ok=True)

    out_m7 = os.path.join(out_dir, "s2_-7_std_512.tif")

    # 已存在就跳过（你如果要强制重做，把这段删掉）
    if os.path.exists(out_m7) and os.path.getsize(out_m7) > 0:
        return plume_id, True, f"skipped_exists:{out_m7}"

    plume_bounds = [lon - DELTA_DEG, lat - DELTA_DEG, lon + DELTA_DEG, lat + DELTA_DEG]

    img = build_multiband_512_from_product(product_dir, plume_bounds, bug)
    if img is None:
        return plume_id, False, ";".join(bug)

    try:
        # 这里直接按 std512 的写法写出“真多band tif”
        write_gdal_multiband_tif(out_m7, img.astype(np.float32))
    except Exception as e:
        bug.append(f"write_err:{type(e).__name__}:{e}")
        return plume_id, False, ";".join(bug)

    return plume_id, True, ";".join(bug) if bug else ""

# =========================
# Main
# =========================
if __name__ == "__main__":
    debug(f"load manifest: {MANIFEST_CSV}")
    df = pd.read_csv(MANIFEST_CSV)

    debug(f"load merged_file: {MERGED_CSV}")
    dfm = pd.read_csv(MERGED_CSV, usecols=["plume_id", "plume_latitude", "plume_longitude"])

    # merge lat/lon into manifest
    df["plume_id"] = df["plume_id"].astype(str)
    dfm["plume_id"] = dfm["plume_id"].astype(str)
    df = df.merge(dfm, on="plume_id", how="left", suffixes=("", "_m"))
    # 统一经纬度列名（优先用 merged_file 里的）
    if "plume_latitude_m" in df.columns:
        df["plume_latitude"] = df["plume_latitude_m"]
    if "plume_longitude_m" in df.columns:
        df["plume_longitude"] = df["plume_longitude_m"]
    # 清理临时列
    for c in ["plume_latitude_m", "plume_longitude_m"]:
        if c in df.columns:
            df.drop(columns=[c], inplace=True)


    # need these columns in manifest
    need = ["plume_id", "raw_dir_exists", "s2_minus7_product_name"]
    for c in need:
        if c not in df.columns:
            raise RuntimeError(f"manifest missing column: {c}")

    # add output columns
    if "s2_-7_std_512" not in df.columns:
        df["s2_-7_std_512"] = ""
    if "m7_std_ok" not in df.columns:
        df["m7_std_ok"] = 0
    if "m7_bug" not in df.columns:
        df["m7_bug"] = ""

    # target rows: already downloaded raw dir
    tgt = df[df["raw_dir_exists"] == 1].copy()
    debug(f"rows to rebuild -7 std512 = {len(tgt)}")

    ok = fail = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {}
        for _, r in tgt.iterrows():
            pid = str(r["plume_id"])
            pname = str(r["s2_minus7_product_name"]) if isinstance(r["s2_minus7_product_name"], str) else ""
            lat = r.get("plume_latitude", np.nan)
            lon = r.get("plume_longitude", np.nan)

            if not np.isfinite(lat) or not np.isfinite(lon):
                df.loc[df["plume_id"] == pid, "m7_std_ok"] = 0
                df.loc[df["plume_id"] == pid, "m7_bug"] = "missing_latlon"
                continue

            futs[ex.submit(process_one, pid, pname, float(lat), float(lon))] = pid

        for fut in as_completed(futs):
            pid = futs[fut]
            try:
                pid2, is_ok, bug = fut.result()
            except Exception as e:
                pid2, is_ok, bug = pid, False, f"worker_exception:{type(e).__name__}:{e}"

            if is_ok:
                ok += 1
                out_path = os.path.join(OUT_ROOT, pid2, "s2_-7_std_512.tif")
                df.loc[df["plume_id"] == pid2, "s2_-7_std_512"] = out_path
                df.loc[df["plume_id"] == pid2, "m7_std_ok"] = 1
                df.loc[df["plume_id"] == pid2, "m7_bug"] = bug
                if bug:
                    debug(f"[{pid2}] OK but BUG: {bug}")
            else:
                fail += 1
                df.loc[df["plume_id"] == pid2, "m7_std_ok"] = 0
                df.loc[df["plume_id"] == pid2, "m7_bug"] = bug
                debug(f"[{pid2}] FAIL bug={bug}")

    debug(f"DONE ok={ok} fail={fail}")

    # overwrite manifest (or change to a new path if you prefer)
    tmp = MANIFEST_CSV + ".part"
    df.to_csv(tmp, index=False)
    os.replace(tmp, MANIFEST_CSV)
    debug(f"updated manifest saved: {MANIFEST_CSV}")



[2026-01-17 00:49:43][pid:944670][tid:139838567121984] load manifest: /data2/yuyao/methane_emission/preprocess_dataset_s2/manifest_minus7_plume_to_safe.csv
[2026-01-17 00:49:43][pid:944670][tid:139838567121984] load merged_file: /data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file.csv
[2026-01-17 00:49:43][pid:944670][tid:139838567121984] rows to rebuild -7 std512 = 4295
[2026-01-17 00:49:44][pid:944670][tid:139838567121984] [GAO20191020t173137p0000-A] OK but BUG: skipped_exists:/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_-790360_512/GAO20191020t173137p0000-A/s2_-7_std_512.tif
[2026-01-17 00:49:44][pid:944670][tid:139838567121984] [GAO20191020t163829p0000-D] OK but BUG: skipped_exists:/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_-790360_512/GAO20191020t163829p0000-D/s2_-7_std_512.tif
[2026-01-17 00:49:44][pid:944670][tid:139838567121984] [GAO20191020t165717p0000-D] OK but BUG: skipped_exists:/mnt/engg-leung/Researc

/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
/home/yuyao/m

[2026-01-17 00:49:57][pid:944670][tid:139838567121984] [GAO20200714t184652p0000-E] OK but BUG: band12_not512:(512, 306);band8A_not512:(512, 306);band04_not512:(512, 306);band02_not512:(512, 306);band07_not512:(512, 306);band03_not512:(512, 306);band11_not512:(512, 306);band06_not512:(512, 306);band05_not512:(512, 306);band01_not512:(512, 306);bands_missing:0/10 missing=[1, 2, 3, 4, 5, 6, 7, 8, 11, 12]


/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


[2026-01-17 00:49:58][pid:944670][tid:139838567121984] [GAO20200719t150529p0000-C] OK but BUG: band12_not512:(512, 467);band8A_not512:(512, 467);band04_not512:(512, 467);band02_not512:(512, 467);band07_not512:(512, 467);band03_not512:(512, 467);band11_not512:(512, 467);band06_not512:(512, 467);band05_not512:(512, 467);band01_not512:(512, 467);bands_missing:0/10 missing=[1, 2, 3, 4, 5, 6, 7, 8, 11, 12]


...

[2026-01-17 01:09:26][pid:944670][tid:139838567121984] [tan20251021t050308c85s4001-B] OK but BUG: band03_not512:(187, 512);band05_not512:(187, 512);band12_not512:(187, 512);band06_not512:(187, 512);band02_not512:(187, 512);band11_not512:(187, 512);band8A_not512:(187, 512);band04_not512:(187, 512);band01_not512:(187, 512);band07_not512:(187, 512);bands_missing:0/10 missing=[1, 2, 3, 4, 5, 6, 7, 8, 11, 12]
[2026-01-17 01:09:26][pid:944670][tid:139838567121984] [tan20251021t050308c85s4001-C] OK but BUG: band03_not512:(259, 512);band05_not512:(259, 512);band12_not512:(259, 512);band06_not512:(259, 512);band02_not512:(259, 512);band11_not512:(259, 512);band8A_not512:(259, 512);band04_not512:(259, 512);band01_not512:(259, 512);band07_not512:(259, 512);bands_missing:0/10 missing=[1, 2, 3, 4, 5, 6, 7, 8, 11, 12]


/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


[2026-01-17 01:09:26][pid:944670][tid:139838567121984] [tan20251019t185503c73s4001-B] OK but BUG: band11_not512:(407, 512);band05_not512:(407, 512);band04_not512:(407, 512);band07_not512:(407, 512);band02_not512:(407, 512);band03_not512:(407, 512);band06_not512:(407, 512);band01_not512:(407, 512);band8A_not512:(407, 512);band12_not512:(407, 512);bands_missing:0/10 missing=[1, 2, 3, 4, 5, 6, 7, 8, 11, 12]


/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


[2026-01-17 01:09:29][pid:944670][tid:139838567121984] DONE ok=4295 fail=0
[2026-01-17 01:09:30][pid:944670][tid:139838567121984] updated manifest saved: /data2/yuyao/methane_emission/preprocess_dataset_s2/manifest_minus7_plume_to_safe.csv


In [5]:
# import tifffile
# img = tifffile.imread("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_-790360_512/tan20250407t064610c00s4001-A/s2_-7_std_512.tif")
# print(img.shape)

import rasterio
with rasterio.open("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_-790360_512/tan20250407t064610c00s4001-A/s2_-7_std_512.tif") as ds:
    print(ds.count, ds.width, ds.height)
    for i in range(1, ds.count + 1):
        band = ds.read(i)
        print(f"Band {i}: min={band.min()} max={band.max()} mean={band.mean()}")

12 512 512
Band 1: min=5918.0 max=10979.0 mean=9556.48046875
Band 2: min=4090.0 max=13141.0 mean=9802.2490234375
Band 3: min=3632.0 max=13475.0 mean=9725.34765625
Band 4: min=3119.0 max=13887.0 mean=9809.7705078125
Band 5: min=3185.0 max=13729.0 mean=9939.400390625
Band 6: min=3209.0 max=13039.0 mean=9682.810546875
Band 7: min=3157.0 max=13074.0 mean=9463.0703125
Band 8: min=2994.0 max=12367.0 mean=9150.443359375
Band 9: min=0.0 max=0.0 mean=0.0
Band 10: min=0.0 max=0.0 mean=0.0
Band 11: min=1422.0 max=5286.0 mean=2198.51025390625
Band 12: min=1412.0 max=8490.0 mean=2251.882080078125


In [ ]:
# 1 crop raw到512并保存为多波段tif, 正确版本
import os
import time
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import contextlib
from collections import Counter, deque

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_origin
import tifffile
from tqdm import tqdm

# =========================
# Config (EDIT THESE)
# =========================
CM_CSV  = "/data2/yuyao/methane_emission/preprocess_dataset_s2/CM_S2_L2A_gee90360.csv"
OUT_CSV = "/data2/yuyao/methane_emission/preprocess_dataset_s2/CM_S2_L2A_gee90360_std512.csv"

# 输出 4 张 512 标准化图到每个 plume 目录下
OUT_ROOT = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_-790360_512"

WINDOW_SIZE = 512
MAX_WORKERS = 12
FLUSH_EVERY_SEC = 60

# 如果缺 -90/-360，不算 fail（只记 bug）；但 t0 和 -7 必须成功
REQUIRE_T0 = True
REQUIRE_M7 = True

# =========================
# Utils
# =========================
def debug(msg: str) -> None:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}][pid:{os.getpid()}][tid:{threading.get_ident()}] {msg}", flush=True)

def safe_mkdir(p: str) -> None:
    os.makedirs(p, exist_ok=True)

@contextlib.contextmanager
def suppress_stderr():
    # 吞掉 GDAL/OpenJPEG stderr 噪音（但我们会在异常时自己 debug 打印）
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stderr(devnull):
            yield

# =========================
# Array helpers -> BHW standard
# =========================
def to_bhw(arr: np.ndarray) -> np.ndarray:
    """
    Accepts:
      - (H,W)   -> (1,H,W)
      - (B,H,W) -> unchanged
      - (H,W,B) -> transpose to (B,H,W)
    """
    if arr.ndim == 2:
        return arr[None, :, :]
    if arr.ndim != 3:
        raise ValueError(f"Unexpected ndim={arr.ndim}, shape={arr.shape}")

    # if first dim looks like bands
    if arr.shape[0] in (1, 3, 4, 12, 13):
        return arr
    # if last dim looks like bands
    if arr.shape[-1] in (1, 3, 4, 12, 13):
        return np.transpose(arr, (2, 0, 1))
    # fallback: assume already BHW
    return arr

def center_crop_or_pad_bhw(bhw: np.ndarray, out_size=512, pad_if_smaller=True):
    """
    returns: (out_bhw, (x0,y0), note)
    """
    B, H, W = bhw.shape
    if H == out_size and W == out_size:
        return bhw, (0, 0), "noop"

    if H < out_size or W < out_size:
        if not pad_if_smaller:
            return None, None, f"too_small {H}x{W}"
        out = np.zeros((B, out_size, out_size), dtype=bhw.dtype)
        y0 = max(0, (out_size - H) // 2)
        x0 = max(0, (out_size - W) // 2)
        out[:, y0:y0+H, x0:x0+W] = bhw
        return out, (x0, y0), f"pad_center {H}x{W} -> {out_size}x{out_size}"

    # larger -> center crop
    y0 = (H - out_size) // 2
    x0 = (W - out_size) // 2
    out = bhw[:, y0:y0+out_size, x0:x0+out_size]
    return out, (x0, y0), f"crop_center {H}x{W} -> {out_size}x{out_size}"

def center_window(ds, size=512):
    if ds.width < size or ds.height < size:
        return None
    col0 = (ds.width  - size) // 2
    row0 = (ds.height - size) // 2
    return Window(col0, row0, size, size)

# =========================
# Write as "GDAL true multiband"
# =========================
def write_gdal_multiband_tif(out_path: str, bhw: np.ndarray):
    """
    Write BHW array to a real multi-band GeoTIFF readable by rasterio/GDAL as count=B.
    Uses a dummy transform (no CRS needed for ML). Ensures parent dir exists.
    """
    bhw = np.asarray(bhw)
    if bhw.ndim != 3:
        raise ValueError(f"write expects BHW, got shape={bhw.shape}")

    B, H, W = bhw.shape
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    transform = from_origin(0, 0, 1, 1)  # dummy
    profile = {
        "driver": "GTiff",
        "height": H,
        "width": W,
        "count": B,
        "dtype": str(bhw.dtype),
        "transform": transform,
        "compress": "deflate",
        "predictor": 2 if np.issubdtype(bhw.dtype, np.floating) else 1,
        "tiled": True,
        "blockxsize": 256 if W >= 256 else W,
        "blockysize": 256 if H >= 256 else H,
        "BIGTIFF": "IF_SAFER",
    }
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(bhw)

# =========================
# Read -> std512 -> write
# =========================
def std_to_512(in_path: str, out_path: str, tag: str, bug_list: list):
    """
    - t0 (s2.tif): force tifffile read (TIFF stack/pages) to preserve all bands.
    - others: rasterio first (fast window), fallback to tifffile.
    - write output ALWAYS as "GDAL true multiband" GeoTIFF via rasterio writer.
    """
    if not (isinstance(in_path, str) and len(in_path) > 0 and os.path.exists(in_path)):
        bug_list.append(f"{tag}_missing:{in_path}")
        return False, f"{tag}_missing"

    # 快速 skip：已有输出（但确保不是空文件）
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return True, "skipped_exists"

    # ✅ FIX 1: t0 强制 tifffile 读，避免 rasterio 把 TIFF stack 当 1 band
    force_tifffile = (tag == "t0")

    # 1) rasterio path (skip for t0)
    if not force_tifffile:
        try:
            with suppress_stderr():
                with rasterio.open(in_path) as ds:
                    win = center_window(ds, WINDOW_SIZE)
                    if win is None:
                        arr = ds.read()  # (B,H,W)
                    else:
                        arr = ds.read(window=win)  # (B,512,512) ideally

            bhw = arr.astype(np.float32)
            # rasterio 已经是 BHW，但这里保险做一次 to_bhw（避免某些 driver 返回 HWC）
            bhw = to_bhw(bhw)

            bhw2, _, note = center_crop_or_pad_bhw(bhw, out_size=WINDOW_SIZE, pad_if_smaller=True)
            if bhw2 is None:
                bug_list.append(f"{tag}_crop_fail_rasterio:{in_path}")
                return False, f"{tag}_crop_fail"

            # ✅ FIX 2: 写成“GDAL 真·多 band”
            write_gdal_multiband_tif(out_path, bhw2)

            if note != "noop":
                bug_list.append(f"{tag}:{note}")
            return True, ""
        except Exception as e1:
            bug_list.append(f"{tag}_rasterio_err:{type(e1).__name__}:{e1}")

    # 2) tifffile path (fallback or forced)
    try:
        arr = tifffile.imread(in_path)
        bhw = to_bhw(arr).astype(np.float32)
        bhw2, _, note = center_crop_or_pad_bhw(bhw, out_size=WINDOW_SIZE, pad_if_smaller=True)
        if bhw2 is None:
            bug_list.append(f"{tag}_crop_fail_tifffile:{in_path}")
            return False, f"{tag}_crop_fail"

        # ✅ FIX 2: 写成“GDAL 真·多 band”
        write_gdal_multiband_tif(out_path, bhw2)

        if note != "noop":
            bug_list.append(f"{tag}:{note}")
        return True, ""
    except Exception as e2:
        bug_list.append(f"{tag}_tifffile_err:{type(e2).__name__}:{e2}")
        return False, f"{tag}_read_fail"

# =========================
# One-row worker (CM-only)
# =========================
def process_one_row(row: dict):
    plume_id = str(row.get("plume_id", "")).strip()
    if not plume_id:
        return "UNKNOWN", {"std_ok": 0, "std_reason": "missing plume_id", "bug": "missing plume_id"}

    # CM columns
    p0   = row.get("s2_path", "")
    pm7  = row.get("s2_-7_path", "")
    pm90 = row.get("s2_pre_path(-90)", "")
    pm360= row.get("s2_pre_pre_path (-360)", "")

    out_dir = os.path.join(OUT_ROOT, plume_id)
    safe_mkdir(out_dir)

    out_t0   = os.path.join(out_dir, "s2_0_std_512.tif")
    out_m7   = os.path.join(out_dir, "s2_-7_std_512.tif")
    out_m90  = os.path.join(out_dir, "s2_-90_std_512.tif")
    out_m360 = os.path.join(out_dir, "s2_-360_std_512.tif")

    # 断点续跑：四个都存在就跳过
    if all(os.path.exists(p) and os.path.getsize(p) > 0 for p in [out_t0, out_m7, out_m90, out_m360]):
        return plume_id, {
            "std_ok": 1, "std_reason": "skipped_all_exist", "bug": "",
            "s2_0_std_512": out_t0,
            "s2_-7_std_512": out_m7,
            "s2_-90_std_512": out_m90,
            "s2_-360_std_512": out_m360,
            "has_m90": 1,
            "has_m360": 1,
        }

    bug = []

    # t0 (force tifffile read)
    ok0, why0 = std_to_512(p0, out_t0, "t0", bug)
    if REQUIRE_T0 and not ok0:
        return plume_id, {
            "std_ok": 0,
            "std_reason": f"t0_failed:{why0}",
            "bug": "; ".join(bug),
        }

    # -7
    ok7, why7 = std_to_512(pm7, out_m7, "m7", bug)
    if REQUIRE_M7 and not ok7:
        return plume_id, {
            "std_ok": 0,
            "std_reason": f"m7_failed:{why7}",
            "bug": "; ".join(bug),
        }

    # -90 / -360（不强制）
    ok90, _ = std_to_512(pm90, out_m90, "m90", bug)
    ok360,_ = std_to_512(pm360, out_m360, "m360", bug)

    reason = "" if len(bug) == 0 else "; ".join(bug)
    return plume_id, {
        "std_ok": 1,
        "std_reason": reason,
        "bug": reason,
        "s2_0_std_512": out_t0 if os.path.exists(out_t0) else "",
        "s2_-7_std_512": out_m7 if os.path.exists(out_m7) else "",
        "s2_-90_std_512": out_m90 if os.path.exists(out_m90) else "",
        "s2_-360_std_512": out_m360 if os.path.exists(out_m360) else "",
        "has_m90": int(ok90 and os.path.exists(out_m90) and os.path.getsize(out_m90) > 0),
        "has_m360": int(ok360 and os.path.exists(out_m360) and os.path.getsize(out_m360) > 0),
    }

# =========================
# Main
# =========================
if __name__ == "__main__":
    debug(f"load CM: {CM_CSV}")
    df = pd.read_csv(CM_CSV)

    need_cols = ["plume_id", "s2_path", "s2_-7_path", "s2_pre_path(-90)", "s2_pre_pre_path (-360)"]
    for c in need_cols:
        if c not in df.columns:
            raise RuntimeError(f"CM missing column: {c}")

    out_cols = [
        "std_ok", "std_reason", "bug",
        "s2_0_std_512", "s2_-7_std_512", "s2_-90_std_512", "s2_-360_std_512",
        "has_m90", "has_m360",
    ]
    for c in out_cols:
        if c not in df.columns:
            df[c] = ""

    # 只处理未完成的
    work_df = df[df["std_ok"].astype(str) != "1"].copy()
    debug(f"rows to process = {len(work_df)}")

    df_lock = threading.Lock()
    flush_state = {"last": time.time()}

    def flush_if_needed(force=False):
        now = time.time()
        if force or (now - flush_state["last"] >= FLUSH_EVERY_SEC):
            tmp = OUT_CSV + ".part"
            with df_lock:
                df.to_csv(tmp, index=False)
                os.replace(tmp, OUT_CSV)
            flush_state["last"] = now
            debug(f"flushed: {OUT_CSV}")

    ok = 0
    fail = 0
    skip = 0
    start = time.time()

    recent = deque(maxlen=200)
    fail_reasons = Counter()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = [ex.submit(process_one_row, r.to_dict()) for _, r in work_df.iterrows()]

        for j, fut in enumerate(tqdm(as_completed(futs), total=len(futs), desc="CM std/crop to 512", mininterval=1.0), start=1):
            try:
                pid, upd = fut.result()
            except Exception as e:
                pid, upd = "UNKNOWN", {"std_ok": 0, "std_reason": f"worker_exception:{e}", "bug": str(e)}

            is_ok = int(upd.get("std_ok", 0)) == 1
            reason = str(upd.get("std_reason", ""))

            # --- 实时打印：失败一定打；OK 但有 bug 也打
            if not is_ok:
                debug(f"[{pid}] FAIL reason={reason} | bug={upd.get('bug')}")
                fail += 1
                recent.append(0)
                fail_reasons[reason.split(";")[0][:120]] += 1
            else:
                if reason == "skipped_all_exist":
                    skip += 1
                else:
                    ok += 1
                recent.append(1)
                if upd.get("bug"):
                    debug(f"[{pid}] OK but BUG: {upd.get('bug')}")

            # --- 写回 df
            with df_lock:
                m = (df["plume_id"].astype(str) == str(pid))
                for k, v in upd.items():
                    if k in df.columns:
                        df.loc[m, k] = v

            # --- 每200条统计 + flush
            if (j % 200) == 0:
                recent_ok = sum(recent) / max(1, len(recent))
                elapsed_min = (time.time() - start) / 60
                top_fail = fail_reasons.most_common(5)
                debug(f"[STAT] done={j}/{len(work_df)} ok={ok} fail={fail} skip={skip} recent200_ok={recent_ok:.2%} elapsed={elapsed_min:.1f} min")
                debug(f"[STAT] top_fail_reasons={top_fail}")
                flush_if_needed(False)

    flush_if_needed(True)
    elapsed_min = (time.time() - start) / 60
    debug(f"DONE ok={ok} fail={fail} skip={skip} elapsed={elapsed_min:.1f} min")

[2026-01-16 18:15:07][pid:374297][tid:140068808557632] load CM: /data2/yuyao/methane_emission/preprocess_dataset_s2/CM_S2_L2A_gee90360.csv
[2026-01-16 18:15:07][pid:374297][tid:140068808557632] rows to process = 4366


CM std/crop to 512:   0%|          | 0/4366 [00:00<?, ?it/s]

[2026-01-16 18:15:08][pid:374297][tid:140068808557632] [STAT] done=200/4366 ok=0 fail=0 skip=200 recent200_ok=100.00% elapsed=0.0 min
[2026-01-16 18:15:08][pid:374297][tid:140068808557632] [STAT] top_fail_reasons=[]
[2026-01-16 18:15:08][pid:374297][tid:140068808557632] [GAO20200805t182912p0000-A] FAIL reason=m7_failed:m7_missing | bug=m7_missing:nan
[2026-01-16 18:15:08][pid:374297][tid:140068808557632] [STAT] done=400/4366 ok=0 fail=1 skip=399 recent200_ok=99.50% elapsed=0.0 min
[2026-01-16 18:15:08][pid:374297][tid:140068808557632] [STAT] top_fail_reasons=[('m7_failed:m7_missing', 1)]
[2026-01-16 18:15:08][pid:374297][tid:140068808557632] [GAO20210508t153858p0000-1] FAIL reason=m7_failed:m7_missing | bug=m7_missing:nan
[2026-01-16 18:15:08][pid:374297][tid:140068808557632] [GAO20210518t152803p0000-1] FAIL reason=m7_failed:m7_missing | bug=m7_missing:nan
[2026-01-16 18:15:08][pid:374297][tid:140068808557632] [GAO20210521t131640p0000-2] FAIL reason=m7_failed:m7_missing | bug=m7_missin

CM std/crop to 512:  14%|█▍        | 632/4366 [00:01<00:05, 631.55it/s]

[2026-01-16 18:15:08][pid:374297][tid:140068808557632] [STAT] done=800/4366 ok=0 fail=38 skip=762 recent200_ok=100.00% elapsed=0.0 min
[2026-01-16 18:15:08][pid:374297][tid:140068808557632] [STAT] top_fail_reasons=[('m7_failed:m7_missing', 38)]
[2026-01-16 18:15:09][pid:374297][tid:140068808557632] [GAO20211004t162512p0000-I] FAIL reason=m7_failed:m7_missing | bug=m7_missing:nan
[2026-01-16 18:15:09][pid:374297][tid:140068808557632] [STAT] done=1000/4366 ok=0 fail=39 skip=961 recent200_ok=99.50% elapsed=0.0 min
[2026-01-16 18:15:09][pid:374297][tid:140068808557632] [STAT] top_fail_reasons=[('m7_failed:m7_missing', 39)]
[2026-01-16 18:15:09][pid:374297][tid:140068808557632] [STAT] done=1200/4366 ok=0 fail=39 skip=1161 recent200_ok=100.00% elapsed=0.0 min
[2026-01-16 18:15:09][pid:374297][tid:140068808557632] [STAT] top_fail_reasons=[('m7_failed:m7_missing', 39)]
[2026-01-16 18:15:09][pid:374297][tid:140068808557632] [GAO20220921t172513p0000-A] OK but BUG: m90_missing:nan; m360_missing:n

...

CM std/crop to 512:  93%|█████████▎| 4066/4366 [00:08<00:00, 435.58it/s]

[2026-01-16 18:15:16][pid:374297][tid:140068808557632] [tan20250729t093434c54s4001-A] FAIL reason=m7_failed:m7_missing | bug=m7_missing:nan
[2026-01-16 18:15:16][pid:374297][tid:140068808557632] [tan20250730t181831c58s4001-Q] FAIL reason=m7_failed:m7_missing | bug=m7_missing:nan
[2026-01-16 18:15:16][pid:374297][tid:140068808557632] [tan20250803t171011c58s4001-A] FAIL reason=m7_failed:m7_missing | bug=m7_missing:nan
[2026-01-16 18:15:16][pid:374297][tid:140068808557632] [tan20250802t113606c33s4001-A] FAIL reason=m7_failed:m7_missing | bug=m7_missing:nan
[2026-01-16 18:15:16][pid:374297][tid:140068808557632] [tan20250803t171011c58s4001-B] FAIL reason=m7_failed:m7_missing | bug=m7_missing:nan
[2026-01-16 18:15:16][pid:374297][tid:140068808557632] [tan20250708t142356c33s4001-A] OK but BUG: m90_rasterio_err:ValueError:could not broadcast input array from shape (12,514,510) into shape (12,512,510); m90_tifffile_err:ValueError:could not broadcast input array from shape (12,514,510) into shap

CM std/crop to 512: 100%|██████████| 4366/4366 [00:09<00:00, 459.78it/s]


[2026-01-16 18:15:17][pid:374297][tid:140068808557632] flushed: /data2/yuyao/methane_emission/preprocess_dataset_s2/CM_S2_L2A_gee90360_std512.csv
[2026-01-16 18:15:17][pid:374297][tid:140068808557632] DONE ok=43 fail=174 skip=4149 elapsed=0.2 min


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/ipykernel/utils.py", line 60, in run_in_context
    return await asyncio.create_task(coro, context=context)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/ipykernel/kernelbase.py", line 614, in shell_main
    await self.dispatch_shell(msg, subshell_id=subshell_id)
  File "/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/site-packages/ipykernel/kernelbase.py", line 486, in dispatch_shell
    sys.stderr.flush()
ValueError: I/O operation on closed file.


In [ ]:
# 2 dropna
import pandas as pd
all_csv = '/data2/yuyao/methane_emission/preprocess_dataset_s2/CM_S2_L2A_gee90360_std512.csv'
df = pd.read_csv(all_csv)
df=df.dropna(subset=['s2_-360_std_512','s2_-90_std_512','s2_-7_std_512','s2_0_std_512'])
print(len(df))
df.to_csv('/data2/yuyao/methane_emission/preprocess_dataset_s2/CM_S2_L2A_gee90360_std512.csv',index=False)

4162


In [12]:
# 3 划分训练集和测试集
import pandas as pd
all_csv = '/data2/yuyao/methane_emission/preprocess_dataset_s2/CM_S2_L2A_-7_gee90360_std512.csv'
df = pd.read_csv(all_csv)
df['date'] = pd.to_datetime(df['datetime'])
# train_df = df[(df['date'] < '2024-04-10') | (df['date'] >= '2025-01-01')]  # 2025-04-01
# test_df = df[(df['date'] >= '2024-04-10') & (df['date'] < '2025-01-01')]
# train_df = df[(df['date'] < '2024-05-10')]  # 2025-04-01
# test_df = df[(df['date'] >= '2024-05-10') & (df['date'] < '2024-12-31')]
train_df = df[(df['date'] < '2025-4-10')]  # 2025-04-01
test_df = df[(df['date'] >= '2025-4-10')]
print(f'training set {len(train_df)} testing set {len(test_df)} train ratio {len(train_df)/(len(train_df)+len(test_df)):.3f}')

train_csv = '/data2/yuyao/methane_emission/data_csv/s2_-790360_temporal_CDSE0_gee90360_2025/train.csv'
train_df.to_csv(train_csv, index=False)
print(f'train set size {len(train_df)}')

test_csv = '/data2/yuyao/methane_emission/data_csv/s2_-790360_temporal_CDSE0_gee90360_2025/test.csv'
test_df.to_csv(test_csv, index=False)
print(f'test set size {len(test_df)}')

training set 3306 testing set 760 train ratio 0.813
train set size 3306
test set size 760


In [ ]:
# 4 从处理好的数据集中裁剪出训练样本16个正样本+16个负样本，这里是0_90_360版本
import os
import pandas as pd
import numpy as np
import tifffile
import random
import threading
from concurrent.futures import ThreadPoolExecutor
import math
import imagecodecs  # keep if you need it

base_dir = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2025_16"
train_csv_path = os.path.join(base_dir, "train.csv")
test_csv_path = os.path.join(base_dir, "test.csv")

origin_train_csv = "/data2/yuyao/methane_emission/data_csv/s2_-790360_temporal_CDSE0_gee90360_2025/train.csv"
origin_test_csv = "/data2/yuyao/methane_emission/data_csv/s2_-790360_temporal_CDSE0_gee90360_2025/test.csv"

os.makedirs(base_dir, exist_ok=True)

image_size = 32
band_index = 11  # 第12波段 (0-based in CHW)
zero_ratio_thresh = 0.20

N_POS = 16
N_NEG = 16

def to_chw(data):
    if data.ndim == 2:
        return data[None, :, :]
    if data.ndim != 3:
        return None
    # 更稳：优先用 bands=12/13 判断
    if data.shape[0] in (12, 13):        # CHW
        return data
    if data.shape[-1] in (12, 13):       # HWC
        return data.transpose(2, 0, 1)
    # fallback（尽量不炸）
    if data.shape[0] <= 20 and data.shape[0] < data.shape[-1]:
        return data
    return data.transpose(2, 0, 1)

def get_crop(width=512, height=512, center_size=20, crop_width=128):
    """
    Return (x, y): x = left (col), y = top (row)
    """
    center_x = width // 2
    center_y = height // 2
    center_left = center_x - center_size // 2
    center_right = center_x + center_size // 2
    center_top = center_y - center_size // 2
    center_bottom = center_y + center_size // 2

    left_min = max(0, center_right - crop_width)
    left_max = min(center_left, width - crop_width)
    top_min = max(0, center_bottom - crop_width)
    top_max = min(center_top, height - crop_width)

    x = random.randint(left_min, left_max)
    y = random.randint(top_min, top_max)
    return x, y

def is_center_contained(crop_xy, image_size=32, width=512, height=512, center_size=10):
    """
    crop_xy: (x, y)
    """
    center_x = width // 2
    center_y = height // 2

    cx1 = center_x - center_size // 2
    cy1 = center_y - center_size // 2
    cx2 = center_x + center_size // 2
    cy2 = center_y + center_size // 2

    x, y = crop_xy
    x1, y1 = x, y
    x2, y2 = x + image_size, y + image_size

    return (x1 <= cx1 and y1 <= cy1 and x2 >= cx2 and y2 >= cy2)

class ThreadSafeCounter:
    def __init__(self):
        self.lock = threading.Lock()
        self.counter = 0
    def increment(self):
        with self.lock:
            self.counter += 1
            return self.counter
    def get(self):
        with self.lock:
            return self.counter

def band12_zero_too_much(arr_chw):
    total = arr_chw[band_index].size
    if total == 0:
        return True
    zero_count = np.sum(arr_chw[band_index] == 0)
    return (zero_count / total) >= zero_ratio_thresh

def crop_chw(arr_chw, crop_xy, size):
    """
    arr_chw: (C,H,W)
    crop_xy: (x,y) where x=col, y=row
    """
    x, y = crop_xy
    return arr_chw[:, y:y+size, x:x+size]

def crop_hw(arr_hw, crop_xy, size):
    x, y = crop_xy
    return arr_hw[y:y+size, x:x+size]

def process_row(row, index, total_count, output_cnt, plume_cnt):
    t_data = to_chw(tifffile.imread(row["s2_0_std_512"]))
    t1_data = to_chw(tifffile.imread(row["s2_-90_std_512"]))
    t2_data = to_chw(tifffile.imread(row["s2_-360_std_512"]))
    mask = tifffile.imread(row["resized_512x512_path"])  # (H,W)

    if t_data is None or t1_data is None or t2_data is None or mask is None:
        return []

    data = []

    # 正样本 crop（包含中心区域）
    crop_list = [get_crop(crop_width=image_size) for _ in range(N_POS)]
    for crop_xy in crop_list:
        nt_data = crop_chw(t_data, crop_xy, image_size)
        nt1_data = crop_chw(t1_data, crop_xy, image_size)
        nt2_data = crop_chw(t2_data, crop_xy, image_size)
        n_mask  = crop_hw(mask, crop_xy, image_size)

        # 防止边界/异常导致尺寸不对
        if nt_data.shape[-2:] != (image_size, image_size) or n_mask.shape != (image_size, image_size):
            continue

        if band12_zero_too_much(nt_data) or band12_zero_too_much(nt1_data) or band12_zero_too_much(nt2_data):
            continue

        cnt = output_cnt.increment()
        dir_path = os.path.join(base_dir, str(cnt))
        os.makedirs(dir_path, exist_ok=True)

        nt_path = os.path.join(dir_path, "s2.tif")
        nt1_path = os.path.join(dir_path, "s2_90.tif")
        nt2_path = os.path.join(dir_path, "s2_360.tif")
        n_mask_path = os.path.join(dir_path, "plume.tif")

        tifffile.imwrite(nt_path, nt_data)
        tifffile.imwrite(nt1_path, nt1_data)
        tifffile.imwrite(nt2_path, nt2_data)
        tifffile.imwrite(n_mask_path, n_mask)

        data.append({
            "id": cnt,
            "s2_path": nt_path,
            "s2_pre_path": nt1_path,
            "s2_pre_pre_path": nt2_path,
            "plume_mask_path": n_mask_path,
            "label": 1,
            "latitude": row["plume_latitude"],
            "longitude": row["plume_longitude"],
            "datetime": row["datetime"],
        })
        plume_cnt.increment()

    # 负样本 crop（随机）
    crop_list = [(random.randint(0, 512 - image_size), random.randint(0, 512 - image_size)) for _ in range(N_NEG)]
    for crop_xy in crop_list:
        nt_data = crop_chw(t_data, crop_xy, image_size)
        nt1_data = crop_chw(t1_data, crop_xy, image_size)
        nt2_data = crop_chw(t2_data, crop_xy, image_size)
        n_mask  = crop_hw(mask, crop_xy, image_size)

        if nt_data.shape[-2:] != (image_size, image_size) or n_mask.shape != (image_size, image_size):
            continue

        if band12_zero_too_much(nt_data) or band12_zero_too_much(nt1_data) or band12_zero_too_much(nt2_data):
            continue

        cnt = output_cnt.increment()
        dir_path = os.path.join(base_dir, str(cnt))
        os.makedirs(dir_path, exist_ok=True)

        nt_path = os.path.join(dir_path, "s2.tif")
        nt1_path = os.path.join(dir_path, "s2_90.tif")
        nt2_path = os.path.join(dir_path, "s2_360.tif")

        label = is_center_contained(crop_xy, image_size=image_size)
        if not label:
            n_mask = np.zeros((image_size, image_size), dtype=mask.dtype)
        else:
            plume_cnt.increment()

        n_mask_path = os.path.join(dir_path, "plume.tif")

        tifffile.imwrite(nt_path, nt_data)
        tifffile.imwrite(nt1_path, nt1_data)
        tifffile.imwrite(nt2_path, nt2_data)
        tifffile.imwrite(n_mask_path, n_mask)

        data.append({
            "id": cnt,
            "s2_path": nt_path,
            "s2_pre_path": nt1_path,
            "s2_pre_pre_path": nt2_path,
            "plume_mask_path": n_mask_path,
            "label": 0 if not label else 1,
            "latitude": row["plume_latitude"],
            "longitude": row["plume_longitude"],
            "datetime": row["datetime"],
        })

    print(f"processed {index} / {total_count} plume_cnt {plume_cnt.get()} / {output_cnt.get()}")
    return data

def run_split(csv_path, out_csv_path, output_cnt, plume_cnt):
    org_df = pd.read_csv(csv_path)
    data_all = []

    batch_size = 32
    total_rows = len(org_df)
    num_batches = math.ceil(total_rows / batch_size)

    for batch_num in range(num_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, total_rows)
        batch_df = org_df.iloc[start_idx:end_idx]

        futures = []
        with ThreadPoolExecutor(max_workers=16) as executor:
            for index, row in batch_df.iterrows():
                futures.append(executor.submit(process_row, row, index, total_rows, output_cnt, plume_cnt))
            for future in futures:
                data_all.extend(future.result())

    out_df = pd.DataFrame(data_all)
    out_df.to_csv(out_csv_path, index=False)
    print(f"wrote {out_csv_path}, size={len(out_df)}")
    return out_df

output_cnt = ThreadSafeCounter()
plume_cnt = ThreadSafeCounter()

train_df = run_split(origin_train_csv, train_csv_path, output_cnt, plume_cnt)
test_df = run_split(origin_test_csv, test_csv_path, output_cnt, plume_cnt)

print(f"total count: {len(train_df) + len(test_df)}")
print(f"train label sum: {train_df['label'].sum()} / {len(train_df)}")
print(f"test label sum: {test_df['label'].sum()} / {len(test_df)}")


processed 15 / 3306 plume_cnt 258 / 471
processed 9 / 3306 plume_cnt 258 / 494
processed 13 / 3306 plume_cnt 258 / 497
processed 8 / 3306 plume_cnt 258 / 499
processed 1 / 3306 plume_cnt 258 / 499
processed 11 / 3306 plume_cnt 258 / 500
processed 14 / 3306 plume_cnt 258 / 504
processed 20 / 3306 plume_cnt 259 / 509
processed 6 / 3306 plume_cnt 259 / 510
processed 5 / 3306 plume_cnt 259 / 510
processed 3 / 3306 plume_cnt 259 / 510
processed 0 / 3306 plume_cnt 259 / 515
processed 7 / 3306 plume_cnt 259 / 515
processed 12 / 3306 plume_cnt 259 / 515
processed 10 / 3306 plume_cnt 261 / 517
processed 25 / 3306 plume_cnt 261 / 520
processed 4 / 3306 plume_cnt 261 / 520
processed 2 / 3306 plume_cnt 264 / 523
processed 16 / 3306 plume_cnt 482 / 926
processed 21 / 3306 plume_cnt 482 / 946
processed 17 / 3306 plume_cnt 482 / 947
processed 23 / 3306 plume_cnt 482 / 949
processed 18 / 3306 plume_cnt 482 / 949
processed 19 / 3306 plume_cnt 482 / 950
processed 28 / 3306 plume_cnt 482 / 950
processed 

In [ ]:
# 5 resize all tif images to 224x224
import os
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile as tiff
from concurrent.futures import ProcessPoolExecutor

TRAIN_CSV = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/train.csv"
TEST_CSV  = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/test.csv"

OLD_ROOT = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16"
NEW_ROOT = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16_224"

IMG_COLS = ["image_path", "s2_pre_path", "s2_pre_pre_path"]

def make_new_path(old_path: str) -> str:
    rel = Path(old_path).relative_to(OLD_ROOT)
    return str(Path(NEW_ROOT) / rel)

def resize_bilinear_chw(img: np.ndarray, out_h=224, out_w=224) -> np.ndarray:
    # img: C x H x W
    if img.ndim != 3:
        raise ValueError(f"Expected CHW, got {img.shape}")
    c, h, w = img.shape

    y = np.linspace(0, h - 1, out_h)
    x = np.linspace(0, w - 1, out_w)
    x0 = np.floor(x).astype(np.int32)
    x1 = np.clip(x0 + 1, 0, w - 1)
    y0 = np.floor(y).astype(np.int32)
    y1 = np.clip(y0 + 1, 0, h - 1)

    wx = (x - x0)[None, :]   # 1 x W
    wy = (y - y0)[:, None]   # H x 1

    Ia = img[:, y0[:, None], x0[None, :]]  # C x H x W
    Ib = img[:, y0[:, None], x1[None, :]]
    Ic = img[:, y1[:, None], x0[None, :]]
    Id = img[:, y1[:, None], x1[None, :]]

    out = (Ia * (1 - wx) * (1 - wy) +
           Ib * wx * (1 - wy) +
           Ic * (1 - wx) * wy +
           Id * wx * wy)

    return out.astype(img.dtype)

def resize_tif(src_path: str, dst_path: str):
    dst_path = Path(dst_path)
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    img = tiff.imread(src_path)  # 期望 C x H x W
    # print(img.shape)
    out = resize_bilinear_chw(img, 224, 224)
    tiff.imwrite(dst_path, out)

def _resize_one(p, new_p):
    resize_tif(p, new_p)
    return new_p

def process_csv(csv_path: str, out_csv_path: str, workers=4):
    df = pd.read_csv(csv_path)

    for col in IMG_COLS:
        paths = df[col].astype(str).tolist()
        new_paths = [make_new_path(p) for p in paths]

        with ProcessPoolExecutor(max_workers=workers) as ex:
            list(ex.map(_resize_one, paths, new_paths))

        df[col] = new_paths

    Path(out_csv_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv_path, index=False)

def main():
    out_train = str(Path(NEW_ROOT) / "train.csv")
    out_test  = str(Path(NEW_ROOT) / "test.csv")
    process_csv(TRAIN_CSV, out_train)
    process_csv(TEST_CSV, out_test)

if __name__ == "__main__":
    main()


Exception in thread Thread-4:
Traceback (most recent call last):
  File "/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/concurrent/futures/process.py", line 347, in run
    self.terminate_broken(cause)
  File "/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/concurrent/futures/process.py", line 499, in terminate_broken
    work_item.future.set_exception(bpe)
  File "/home/yuyao/miniconda3/envs/s2geo/lib/python3.11/concurrent/futures/_base.py", line 559, in set_exception
    raise InvalidStateError('{}: {!r}'.format(self._state, self))
concurrent.futures._base.InvalidStateError: CANCELLED: <Future at 0x7f0938433550 state=cancelled>


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [ ]:
# 4-2 crop 32x32 samples from 512x512 images at t0, t-7, t-90, t-360 版本
# crop to 32 (t0 as label=1, t-7 as label=0) — center-box-only labeling (FIXED)
# ============================================================
# Fixes vs your previous t-7 script:
#  1) POS now requires plume pixels in cropped mask (cm.sum() > 0)
#  2) NEG now requires NO plume pixels at t0 mask for same crop (cm_t0.sum() == 0)
#  3) Apply band-zero filtering consistently to image + pre90 + pre360 (like t0 script)
# ============================================================

import os
import math
import random
import threading
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import tifffile

# =========================
# Config (EDIT THESE)
# =========================
TRAIN_CSV  = "/data2/yuyao/methane_emission/data_csv/s2_-790360_temporal_CDSE0_gee90360_2024/train.csv"
TEST_CSV   = "/data2/yuyao/methane_emission/data_csv/s2_-790360_temporal_CDSE0_gee90360_2024/test.csv"

OUT_DIR    = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024_fixed"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

MASK_ROOT = "/data2/yuyao/methane_emission/carbon_mapper_data_masks"

PATCH_SIZE = 32
CENTER_BOX = 256          # STRICT: patch-center must lie in this box (around 256,256)

N_POS = 8
N_NEG = 8
MAX_TRIES_POS = 800       # increased a bit because we now have accept/reject
MAX_TRIES_NEG = 800

MAX_WORKERS = 16
BATCH_SIZE = 32

# band-zero filtering (match your t0 pipeline style)
BAND_INDEX = 11           # B12 (0-based)
ZERO_RATIO_THRESH = 0.20  # recommend same as t0 script

# mask gating thresholds
POS_MASK_MIN_SUM = 1      # >=1 means at least one plume pixel
NEG_MASK_MAX_SUM = 0      # must be 0 plume pixels for negative

# =========================
# Utils
# =========================
def debug(msg: str) -> None:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}][pid:{os.getpid()}][tid:{threading.get_ident()}] {msg}", flush=True)

def to_chw(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 2:
        return arr[None, :, :]
    if arr.ndim != 3:
        raise ValueError(f"Unexpected ndim={arr.ndim}, shape={arr.shape}")
    if arr.shape[0] in (1, 3, 4, 12, 13):     # CHW
        return arr
    if arr.shape[-1] in (1, 3, 4, 12, 13):    # HWC -> CHW
        return np.transpose(arr, (2, 0, 1))
    if arr.shape[0] <= 20 and arr.shape[0] < arr.shape[-1]:
        return arr
    return np.transpose(arr, (2, 0, 1))

def ensure_abs(p):
    if not isinstance(p, str) or not p.strip():
        return ""
    p = p.strip()
    return p if os.path.isabs(p) else os.path.abspath(p)

def band_zero_too_much(chw: np.ndarray) -> bool:
    if chw is None or chw.ndim != 3:
        return True
    if chw.shape[0] <= BAND_INDEX:
        # if no band12 exists, don't filter here (or set True if you want strict)
        return False
    total = chw[BAND_INDEX].size
    if total == 0:
        return True
    zero_ratio = float((chw[BAND_INDEX] == 0).sum()) / float(total)
    return zero_ratio >= ZERO_RATIO_THRESH

def crop_chw(chw: np.ndarray, x: int, y: int, size: int) -> np.ndarray:
    return chw[:, y:y+size, x:x+size]

def crop_hw(hw: np.ndarray, x: int, y: int, size: int) -> np.ndarray:
    return hw[y:y+size, x:x+size]

def sample_center_xy_strict(width=512, height=512, patch_size=32, center_box=256):
    """
    STRICT:
      Sample (x,y) top-left such that patch-center lies inside center_box x center_box
      around (256,256). NEVER samples outside. If impossible -> raise ValueError.
    """
    cx = width // 2
    cy = height // 2
    half_box = center_box // 2

    min_center_x = cx - half_box
    max_center_x = cx + half_box
    min_center_y = cy - half_box
    max_center_y = cy + half_box

    # patch center = (x + patch_size//2, y + patch_size//2)
    min_x = max(0, min_center_x - patch_size // 2)
    max_x = min(width - patch_size, max_center_x - patch_size // 2)
    min_y = max(0, min_center_y - patch_size // 2)
    max_y = min(height - patch_size, max_center_y - patch_size // 2)

    if min_x > max_x or min_y > max_y:
        raise ValueError(
            f"center_box_invalid:min_x={min_x},max_x={max_x},min_y={min_y},max_y={max_y},"
            f"patch={patch_size},center_box={center_box}"
        )

    x = random.randint(int(min_x), int(max_x))
    y = random.randint(int(min_y), int(max_y))
    return x, y

def mask_path_for_plume(plume_id: str) -> str:
    d = os.path.join(MASK_ROOT, plume_id)
    p = os.path.join(d, "resized_512x512.tif")
    return p if os.path.exists(p) else ""

def read_chw_512(path: str, tag: str, bug_list: list):
    path = ensure_abs(path)
    if not path or not os.path.exists(path):
        bug_list.append(f"{tag}_missing:{path}")
        return None
    try:
        arr = tifffile.imread(path)
        chw = to_chw(arr).astype(np.float32)
        if chw.shape[-2:] != (512, 512):
            bug_list.append(f"{tag}_not512:{chw.shape}")
            return None
        return chw
    except Exception as e:
        bug_list.append(f"{tag}_read_err:{type(e).__name__}:{e}")
        return None

def read_mask_512(plume_id: str, bug_list: list):
    mp = mask_path_for_plume(plume_id)
    if not mp:
        bug_list.append("mask_missing")
        return None, ""
    try:
        m = tifffile.imread(mp)
        if m.shape != (512, 512):
            bug_list.append(f"mask_not512:{m.shape}")
            return None, mp
        return m, mp
    except Exception as e:
        bug_list.append(f"mask_read_err:{type(e).__name__}:{e}")
        return None, mp

# =========================
# Per-plume worker
# =========================
def crop_one_plume(row: dict, out_root: str, split_name: str):
    pid = str(row.get("plume_id", "")).strip()
    bug = []

    if not pid:
        return [], "missing_plume_id"

    # required inputs
    p_t0   = ensure_abs(row.get("s2_0_std_512", ""))
    p_m7   = ensure_abs(row.get("s2_-7_std_512", ""))
    p_m90  = ensure_abs(row.get("s2_-90_std_512", ""))
    p_m360 = ensure_abs(row.get("s2_-360_std_512", ""))

    rawvals = (
        f"rawvals:s2_0_std_512={p_t0}|s2_-7_std_512={p_m7}|"
        f"s2_-90_std_512={p_m90}|s2_-360_std_512={p_m360}"
    )

    # strict center-box feasibility check
    try:
        _ = sample_center_xy_strict(512, 512, PATCH_SIZE, CENTER_BOX)
    except Exception as e:
        bug.append(f"center_box_invalid:{e}")
        return [], ";".join(bug) + ";" + rawvals

    # read images
    t0   = read_chw_512(p_t0,   "t0",   bug)
    m7   = read_chw_512(p_m7,   "m7",   bug)
    m90  = read_chw_512(p_m90,  "m90",  bug)
    m360 = read_chw_512(p_m360, "m360", bug)
    if t0 is None or m7 is None or m90 is None or m360 is None:
        return [], ";".join(bug) + ";" + rawvals

    # read mask512 (used for gating + pos plume.tif)
    mask512, mask512_path = read_mask_512(pid, bug)
    if mask512 is None:
        return [], ";".join(bug) + ";" + rawvals + f";mask512={mask512_path}"

    plume_dir = Path(out_root) / split_name / pid
    plume_dir.mkdir(parents=True, exist_ok=True)

    samples = []

    def write_sample(kind: str, k: int, x: int, y: int,
                     img_chw: np.ndarray, pre90_chw: np.ndarray, pre360_chw: np.ndarray,
                     plume_hw: np.ndarray, label: int, source: str):
        d = plume_dir / f"{kind}_{k:02d}_x{x}_y{y}"
        d.mkdir(parents=True, exist_ok=True)

        img_path   = str(d / "image.tif")
        p90_path   = str(d / "s2_90.tif")
        p360_path  = str(d / "s2_360.tif")
        plume_path = str(d / "plume.tif")

        tifffile.imwrite(img_path,   img_chw)
        tifffile.imwrite(p90_path,   pre90_chw)
        tifffile.imwrite(p360_path,  pre360_chw)
        tifffile.imwrite(plume_path, plume_hw)

        samples.append({
            "plume_id": pid,
            "split": split_name,
            "label": int(label),
            "image_path": img_path,
            "s2_pre_path": p90_path,
            "s2_pre_pre_path": p360_path,
            "plume_mask_path": plume_path,
            "crop_x": int(x),
            "crop_y": int(y),
            "source": source,
        })

    # -------------------------
    # POS: t0 label=1
    # STRICT center-box
    # Must contain plume pixels in mask crop
    # -------------------------
    pos_written = 0
    pos_tries = 0
    while pos_written < N_POS and pos_tries < MAX_TRIES_POS:
        pos_tries += 1
        x, y = sample_center_xy_strict(512, 512, PATCH_SIZE, CENTER_BOX)

        c0   = crop_chw(t0,   x, y, PATCH_SIZE)
        c90  = crop_chw(m90,  x, y, PATCH_SIZE)
        c360 = crop_chw(m360, x, y, PATCH_SIZE)
        cm   = crop_hw(mask512, x, y, PATCH_SIZE)

        if c0.shape[-2:] != (PATCH_SIZE, PATCH_SIZE) or cm.shape != (PATCH_SIZE, PATCH_SIZE):
            continue

        # FIX #1: ensure true positive (plume exists)
        if float(cm.sum()) < POS_MASK_MIN_SUM:
            continue

        # FIX #2: filter all frames consistently
        if band_zero_too_much(c0) or band_zero_too_much(c90) or band_zero_too_much(c360):
            continue

        write_sample("pos", pos_written, x, y, c0, c90, c360, cm, label=1, source="t0")
        pos_written += 1

    if pos_written < N_POS:
        bug.append(f"pos_insufficient:{pos_written}/{N_POS}")

    # -------------------------
    # NEG: t-7 label=0
    # STRICT center-box
    # Must be plume-free at t0 mask for same crop (true negative)
    # plume.tif saved all-zero (avoid leakage)
    # -------------------------
    neg_written = 0
    neg_tries = 0
    while neg_written < N_NEG and neg_tries < MAX_TRIES_NEG:
        neg_tries += 1
        x, y = sample_center_xy_strict(512, 512, PATCH_SIZE, CENTER_BOX)

        # gating by t0 mask at same coords
        cm_t0 = crop_hw(mask512, x, y, PATCH_SIZE)
        if cm_t0.shape != (PATCH_SIZE, PATCH_SIZE):
            continue

        # FIX #3: ensure true negative (no plume at t0)
        if float(cm_t0.sum()) > NEG_MASK_MAX_SUM:
            continue

        c7   = crop_chw(m7,   x, y, PATCH_SIZE)
        c90  = crop_chw(m90,  x, y, PATCH_SIZE)
        c360 = crop_chw(m360, x, y, PATCH_SIZE)

        if c7.shape[-2:] != (PATCH_SIZE, PATCH_SIZE):
            continue

        # FIX #4: filter all frames consistently
        if band_zero_too_much(c7) or band_zero_too_much(c90) or band_zero_too_much(c360):
            continue

        cm0 = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=mask512.dtype)
        write_sample("neg", neg_written, x, y, c7, c90, c360, cm0, label=0, source="m7")
        neg_written += 1

    if neg_written < N_NEG:
        bug.append(f"neg_insufficient:{neg_written}/{N_NEG}")

    # require full counts (same as your original script behavior)
    if pos_written < N_POS or neg_written < N_NEG:
        return samples, ";".join(bug) + ";" + rawvals + f";mask512={mask512_path}"

    return samples, ""

# =========================
# Runner
# =========================
def run_split(split_name: str, split_csv: str, out_csv_path: str):
    df = pd.read_csv(split_csv)

    if "plume_id" not in df.columns:
        raise RuntimeError(f"{split_csv} missing plume_id")

    need_cols = ["s2_0_std_512", "s2_-7_std_512", "s2_-90_std_512", "s2_-360_std_512"]
    miss = [c for c in need_cols if c not in df.columns]
    if miss:
        raise RuntimeError(f"{split_csv} missing columns: {miss}")

    debug(f"[{split_name}] rows={len(df)}")

    data_all = []
    total_rows = len(df)
    num_batches = math.ceil(total_rows / BATCH_SIZE)

    ok = 0
    fail = 0

    for b in range(num_batches):
        start_idx = b * BATCH_SIZE
        end_idx = min((b + 1) * BATCH_SIZE, total_rows)
        batch_df = df.iloc[start_idx:end_idx]

        futures = []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            for _, r in batch_df.iterrows():
                futures.append(ex.submit(crop_one_plume, r.to_dict(), OUT_DIR, split_name))

            for fut in futures:
                try:
                    samples, bug = fut.result()
                except Exception as e:
                    samples, bug = [], f"worker_exception:{type(e).__name__}:{e}"

                pid = str(samples[0].get("plume_id", "unknown_plume")) if samples else "unknown_plume"

                if samples and len(samples) >= (N_POS + N_NEG):
                    ok += 1
                    data_all.extend(samples)
                else:
                    fail += 1
                    debug(f"[{split_name}][{pid}] FAIL bug={bug}")

        debug(f"[{split_name}] batch {b+1}/{num_batches} done ok={ok} fail={fail} data_all={len(data_all)}")

    out_df = pd.DataFrame(data_all)
    out_df.to_csv(out_csv_path, index=False)
    debug(f"[{split_name}] wrote {out_csv_path}, size={len(out_df)} ok={ok} fail={fail}")
    return out_df

# =========================
# Main
# =========================
if __name__ == "__main__":
    out_train = os.path.join(OUT_DIR, "train_patches.csv")
    out_test  = os.path.join(OUT_DIR, "test_patches.csv")

    debug("Start cropping train...")
    train_df = run_split("train", TRAIN_CSV, out_train)

    debug("Start cropping test...")
    test_df = run_split("test", TEST_CSV, out_test)

    debug(f"ALL DONE. total={len(train_df) + len(test_df)}")
    if len(train_df) > 0:
        debug(f"train label sum: {int(train_df['label'].sum())} / {len(train_df)}")
    if len(test_df) > 0:
        debug(f"test  label sum: {int(test_df['label'].sum())} / {len(test_df)}")


[2026-01-19 21:25:42][pid:760326][tid:140367905100864] Start cropping train...
[2026-01-19 21:25:42][pid:760326][tid:140367905100864] [train] rows=2559
[2026-01-19 21:25:51][pid:760326][tid:140367905100864] [train][unknown_plume] FAIL bug=pos_insufficient:0/8;neg_insufficient:0/8;rawvals:s2_0_std_512=/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_-790360_512/GAO20191020t171158p0000-C/s2_0_std_512.tif|s2_-7_std_512=/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_-790360_512/GAO20191020t171158p0000-C/s2_-7_std_512.tif|s2_-90_std_512=/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_-790360_512/GAO20191020t171158p0000-C/s2_-90_std_512.tif|s2_-360_std_512=/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_-790360_512/GAO20191020t171158p0000-C/s2_-360_std_512.tif;mask512=/data2/yuyao/methane_emission/carbon_mapper_data_masks/GAO20191020t171158p0000-C/resized_512x512.tif
[2026-01-19 21:25:51][

In [ ]:
# Inspect cropped dataset
import os
import numpy as np
import pandas as pd
import tifffile
from collections import defaultdict

# =========================
# CONFIG
# =========================
# TRAIN_CSV = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024/train_patches.csv"
# TEST_CSV  = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024/test_patches.csv"
TRAIN_CSV = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/train.csv"
TEST_CSV  = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/test.csv"

BAND_INDEX = 11        # Sentinel-2 B12 (0-based)
EPS_STD = 1e-6         # near-constant threshold
MAX_SAMPLES = 2000     # per split (randomly sampled for speed)

# =========================
# Utils
# =========================
def read_chw(path):
    arr = tifffile.imread(path)
    if arr.ndim == 2:
        return arr[None, :, :]
    if arr.ndim == 3 and arr.shape[0] in (1,3,4,12,13):
        return arr
    if arr.ndim == 3:
        return arr.transpose(2,0,1)
    raise ValueError(f"unexpected shape {arr.shape} for {path}")

def safe_stats(x):
    return {
        "mean": float(np.mean(x)),
        "std":  float(np.std(x)),
        "min":  float(np.min(x)),
        "max":  float(np.max(x)),
    }

# =========================
# Core inspection
# =========================
def inspect_split(csv_path, split_name):
    print(f"\n{'='*80}")
    print(f"Inspecting {split_name}: {csv_path}")
    print(f"{'='*80}")

    df = pd.read_csv(csv_path)
    if len(df) > MAX_SAMPLES:
        df = df.sample(MAX_SAMPLES, random_state=42)

    stats = defaultdict(list)
    failures = 0

    for i, row in df.iterrows():
        try:
            img  = read_chw(row["image_path"]).astype(np.float32)
            p90  = read_chw(row["s2_pre_path"]).astype(np.float32)
            p360 = read_chw(row["s2_pre_pre_path"]).astype(np.float32)
            mask = tifffile.imread(row["plume_mask_path"])
        except Exception as e:
            failures += 1
            continue

        label = int(row["label"])

        # ------------------
        # Shape checks
        # ------------------
        stats[(label, "shape")].append(img.shape)

        # ------------------
        # Band-12 checks
        # ------------------
        if img.shape[0] > BAND_INDEX:
            b12 = img[BAND_INDEX]
            zero_ratio = float((b12 == 0).sum()) / b12.size
            stats[(label, "b12_zero_ratio")].append(zero_ratio)
            stats[(label, "b12_std")].append(float(np.std(b12)))

        # ------------------
        # Near-constant image
        # ------------------
        stats[(label, "img_std")].append(float(np.std(img)))

        # ------------------
        # Triplet consistency
        # ------------------
        d90  = float(np.mean(np.abs(img - p90)))
        d360 = float(np.mean(np.abs(img - p360)))
        stats[(label, "diff_90")].append(d90)
        stats[(label, "diff_360")].append(d360)

        # ------------------
        # Mask sanity
        # ------------------
        mask_sum = float(mask.sum())
        stats[(label, "mask_sum")].append(mask_sum)

    # =========================
    # Print summary
    # =========================
    for label in sorted(df["label"].unique()):
        print(f"\n--- Label {label} ---")

        shapes = set(stats[(label, "shape")])
        print(f"shapes: {shapes}")

        if (label, "img_std") in stats:
            near_const = np.mean(np.array(stats[(label, "img_std")]) < EPS_STD)
            print(f"near-constant images: {near_const*100:.2f}%")

        if (label, "b12_zero_ratio") in stats:
            zr = np.array(stats[(label, "b12_zero_ratio")])
            print(f"B12 zero ratio: mean={zr.mean():.3f}, >0.2={(zr>0.2).mean()*100:.1f}%")

        if (label, "diff_90") in stats:
            print(f"|img - s2_90|:  mean={np.mean(stats[(label,'diff_90')]):.4f}")
            print(f"|img - s2_360|: mean={np.mean(stats[(label,'diff_360')]):.4f}")

        if (label, "mask_sum") in stats:
            ms = np.array(stats[(label, "mask_sum")])
            print(f"mask nonzero ratio: {(ms>0).mean()*100:.2f}%")

    print(f"\nRead failures: {failures} / {len(df)}")

# =========================
# Run
# =========================
inspect_split(TRAIN_CSV, "TRAIN")
inspect_split(TEST_CSV,  "TEST")



Inspecting TRAIN: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/train.csv

--- Label 0 ---
shapes: {(12, 32, 32)}
near-constant images: 0.00%
B12 zero ratio: mean=0.000, >0.2=0.0%
|img - s2_90|:  mean=1456.6119
|img - s2_360|: mean=1447.5209
mask nonzero ratio: 0.00%

--- Label 1 ---
shapes: {(12, 32, 32)}
near-constant images: 0.00%
B12 zero ratio: mean=0.000, >0.2=0.0%
|img - s2_90|:  mean=1420.3959
|img - s2_360|: mean=1425.3707
mask nonzero ratio: 100.00%

Read failures: 0 / 2000

Inspecting TEST: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/test.csv

--- Label 0 ---
shapes: {(12, 32, 32)}
near-constant images: 0.00%
B12 zero ratio: mean=0.000, >0.2=0.0%
|img - s2_90|:  mean=1690.3182
|img - s2_360|: mean=1600.4018
mask nonzero ratio: 0.00%

--- Label 1 ---
shapes: {(12, 32, 32)}
near-constant images: 0.00%
B12 zero ratio: mean=0.000, >0.2=0.0%
|img - s2_90|:  mean=1689

In [ ]:
# 扫描32x32 dataset目录，生成train.csv和test.csv
import os
from pathlib import Path
import pandas as pd

# 改这里：你的 32x32 dataset 根目录
OUT_DIR = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024")
def rebuild_split(split: str):
    split_dir = OUT_DIR / split
    assert split_dir.exists(), f"missing split dir: {split_dir}"

    rows = []
    bad = 0

    # 目录结构：split/plume_id/(pos_..../ or neg_..../)/image.tif ...
    for plume_dir in split_dir.iterdir():
        if not plume_dir.is_dir():
            continue
        plume_id = plume_dir.name

        for sample_dir in plume_dir.iterdir():
            if not sample_dir.is_dir():
                continue

            img = sample_dir / "image.tif"
            p90 = sample_dir / "s2_90.tif"
            p360 = sample_dir / "s2_360.tif"
            plume = sample_dir / "plume.tif"

            if not (img.exists() and p90.exists() and p360.exists() and plume.exists()):
                bad += 1
                continue

            name = sample_dir.name
            # label from folder name prefix
            if name.startswith("pos_"):
                label = 1
                source = "t0"
            elif name.startswith("neg_"):
                label = 0
                source = "m7"
            else:
                # unknown naming -> skip
                bad += 1
                continue

            # parse crop_x/crop_y from "..._x123_y456"
            crop_x = crop_y = None
            parts = name.split("_")
            try:
                for i, t in enumerate(parts):
                    if t.startswith("x"):
                        crop_x = int(t[1:])
                    if t.startswith("y"):
                        crop_y = int(t[1:])
            except:
                crop_x = crop_y = None

            rows.append({
                "plume_id": plume_id,
                "split": split,
                "label": int(label),
                "image_path": str(img),
                "s2_pre_path": str(p90),
                "s2_pre_pre_path": str(p360),
                "plume_mask_path": str(plume),
                "crop_x": crop_x,
                "crop_y": crop_y,
                "source": source,
            })

    df = pd.DataFrame(rows)
    out_csv = OUT_DIR / f"{split}.csv"
    df.to_csv(out_csv, index=False)
    print(f"[{split}] wrote {out_csv}  rows={len(df)}  dropped_missing={bad}")
    return df

train_df = rebuild_split("train")
test_df  = rebuild_split("test")

print("done.")
print("train label sum:", int(train_df["label"].sum()), "/", len(train_df))
print("test  label sum:", int(test_df["label"].sum()), "/", len(test_df))


[train] wrote /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024/train.csv  rows=40696  dropped_missing=38974
[test] wrote /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024/test.csv  rows=8128  dropped_missing=0
done.
train label sum: 18105 / 40696
test  label sum: 4056 / 8128


In [22]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024")
df = pd.read_csv(OUT_DIR / "train.csv")

pid = "GAO20191023t152337p0000-A"
pdir = OUT_DIR / "train" / pid
print("dir exists:", pdir.exists(), "is_dir:", pdir.is_dir(), "path:", str(pdir))

# 如果 CSV 里有这个 pid，就把它对应的 image_path 也查一下
r = df[df["plume_id"] == pid].head(1)
if len(r) > 0:
    img_path = Path(r["image_path"].iloc[0])
    print("csv image exists:", img_path.exists(), "path:", str(img_path))


dir exists: False is_dir: False path: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024/train/GAO20191023t152337p0000-A
csv image exists: False path: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024/train/GAO20191023t152337p0000-A/pos_05_x240_y210/image.tif


In [8]:
import pandas as pd

CSV = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024/train_patches.csv"
OUT = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_-790360_32_2024/train.csv"

df = pd.read_csv(CSV)

keep_rows = []

for pid, g in df.groupby("plume_id"):
    pos = g[g["label"] == 1]
    neg = g[g["label"] == 0]

    if len(pos) == 0:
        # 没有 pos 的 plume，全部 neg 丢掉（最干净）
        continue

    # neg 最多保留和 pos 一样多
    neg_keep = neg.sample(
        n=min(len(neg), len(pos)),
        random_state=42
    )

    keep_rows.append(pd.concat([pos, neg_keep], axis=0))

df_bal = pd.concat(keep_rows, axis=0).sample(frac=1, random_state=42)

df_bal.to_csv(OUT, index=False)

print("before:", df["label"].mean(), " size=", len(df))
print("after :", df_bal["label"].mean(), " size=", len(df_bal))


before: 0.5  size= 20816
after : 0.5  size= 20816


In [15]:
import os, random
import numpy as np
import pandas as pd
import tifffile

CSV = "/data2/yuyao/methane_emission/data_csv/s2_-790360_temporal_CDSE0_gee90360_2024/train.csv"
N = 200  # 抽样个数，先 200 足够
BAND_INDEX = 11
ZERO_RATIO_THRESH = 0.20

def to_chw(arr):
    if arr.ndim == 2:
        return arr[None, :, :]
    if arr.ndim != 3:
        return None
    if arr.shape[0] in (12, 13):
        return arr
    if arr.shape[-1] in (12, 13):
        return arr.transpose(2, 0, 1)
    if arr.shape[0] <= 20 and arr.shape[0] < arr.shape[-1]:
        return arr
    return arr.transpose(2, 0, 1)

def read_chw(path):
    if not isinstance(path, str) or not path or (not os.path.exists(path)):
        return None, "missing"
    try:
        arr = tifffile.imread(path)
        chw = to_chw(arr)
        if chw is None:
            return None, f"bad_ndim:{getattr(arr,'shape',None)}"
        if chw.shape[-2:] != (512, 512):
            return None, f"not512:{chw.shape}"
        return chw.astype(np.float32), "ok"
    except Exception as e:
        return None, f"read_err:{e}"

def stats_one(chw):
    # basic stats + zero ratio of band12
    s = {}
    s["nan_frac"] = float(np.isnan(chw).mean())
    s["min"] = float(np.nanmin(chw))
    s["max"] = float(np.nanmax(chw))
    s["mean"] = float(np.nanmean(chw))
    s["std"] = float(np.nanstd(chw))
    if chw.shape[0] > BAND_INDEX:
        b = chw[BAND_INDEX]
        s["b12_zero_ratio"] = float((b == 0).mean())
        s["b12_mean"] = float(np.nanmean(b))
        s["b12_std"] = float(np.nanstd(b))
    else:
        s["b12_zero_ratio"] = None
        s["b12_mean"] = None
        s["b12_std"] = None
    return s

df = pd.read_csv(CSV)
need = ["plume_id", "s2_0_std_512", "s2_-7_std_512"]
for c in need:
    if c not in df.columns:
        raise RuntimeError(f"missing column: {c}")

idx = list(df.index)
random.shuffle(idx)
idx = idx[:min(N, len(idx))]

bad = {"t0_missing":0, "m7_missing":0, "t0_read":0, "m7_read":0, "t0_not512":0, "m7_not512":0, "t0_other":0, "m7_other":0}
rows = []

for i in idx:
    r = df.loc[i]
    pid = str(r["plume_id"])
    p0 = r["s2_0_std_512"]
    p7 = r["s2_-7_std_512"]

    t0, s0 = read_chw(p0)
    m7, s7 = read_chw(p7)

    def count_err(tag, s):
        if s == "ok": return
        if s == "missing": bad[f"{tag}_missing"] += 1
        elif s.startswith("read_err"): bad[f"{tag}_read"] += 1
        elif s.startswith("not512"): bad[f"{tag}_not512"] += 1
        else: bad[f"{tag}_other"] += 1

    count_err("t0", s0)
    count_err("m7", s7)

    if t0 is None or m7 is None:
        continue

    st0 = stats_one(t0)
    sm7 = stats_one(m7)

    rows.append({
        "plume_id": pid,
        "t0_path": p0,
        "m7_path": p7,
        **{f"t0_{k}": v for k, v in st0.items()},
        **{f"m7_{k}": v for k, v in sm7.items()},
        "delta_mean": sm7["mean"] - st0["mean"],
        "delta_std": sm7["std"] - st0["std"],
        "delta_b12_zero_ratio": (sm7["b12_zero_ratio"] - st0["b12_zero_ratio"]) if (sm7["b12_zero_ratio"] is not None and st0["b12_zero_ratio"] is not None) else None
    })

out = pd.DataFrame(rows)
print("=== Read/shape errors (sampled) ===")
print(bad)
print("\n=== Summary over successfully-read pairs ===")
if len(out) == 0:
    print("No valid pairs read.")
else:
    for col in ["t0_nan_frac","m7_nan_frac","t0_std","m7_std","t0_b12_zero_ratio","m7_b12_zero_ratio","delta_b12_zero_ratio","delta_std","delta_mean"]:
        if col in out.columns:
            s = out[col].dropna()
            if len(s):
                print(f"{col:22s}  mean={s.mean():.4g}  p50={s.median():.4g}  p90={s.quantile(0.9):.4g}  max={s.max():.4g}")

# 输出一些最可疑样本，方便你人工 spot check
if len(out):
    # 1) -7 band12 几乎全 0
    sus1 = out.sort_values("m7_b12_zero_ratio", ascending=False).head(10)[["plume_id","m7_b12_zero_ratio","t0_b12_zero_ratio","m7_path","t0_path"]]
    print("\n=== Top suspicious: m7_b12_zero_ratio highest ===")
    print(sus1.to_string(index=False))

    # 2) -7 std 很低（接近常数图）
    sus2 = out.sort_values("m7_std", ascending=True).head(10)[["plume_id","m7_std","t0_std","m7_path","t0_path"]]
    print("\n=== Top suspicious: m7_std lowest ===")
    print(sus2.to_string(index=False))


=== Read/shape errors (sampled) ===
{'t0_missing': 0, 'm7_missing': 0, 't0_read': 0, 'm7_read': 0, 't0_not512': 0, 'm7_not512': 0, 't0_other': 0, 'm7_other': 0}

=== Summary over successfully-read pairs ===
t0_nan_frac             mean=0  p50=0  p90=0  max=0
m7_nan_frac             mean=0  p50=0  p90=0  max=0
t0_std                  mean=842.7  p50=1191  p90=1838  max=2405
m7_std                  mean=1613  p50=1643  p90=1922  max=2486
t0_b12_zero_ratio       mean=0.487  p50=0.1865  p90=1  max=1
m7_b12_zero_ratio       mean=0.008289  p50=0  p90=0  max=0.5572
delta_b12_zero_ratio    mean=-0.4787  p50=-0.002058  p90=0  max=0.5572
delta_std               mean=770.7  p50=135.4  p90=1812  max=2486
delta_mean              mean=1299  p50=211.6  p90=3152  max=3574

=== Top suspicious: m7_b12_zero_ratio highest ===
                  plume_id  m7_b12_zero_ratio  t0_b12_zero_ratio                                                                                                                      

In [13]:
import pandas as pd

TRAIN_CSV = "/data2/yuyao/methane_emission/data_csv/s2_-790360_temporal_CDSE0_gee90360_2024/train.csv"
df = pd.read_csv(TRAIN_CSV)
print(df.columns.tolist())
print(df[["plume_id"]].head(3))
for c in ["s2_0_std_512","s2_-7_std_512","s2_-90_std_512","s2_-360_std_512"]:
    print(c, c in df.columns, df[c].head(3).tolist() if c in df.columns else "NA")

['plume_id', 'plume_latitude', 'plume_longitude', 'datetime', 'ipcc_sector', 'gas', 'emission_cmf_type', 'plume_bounds', 'instrument', 'mission_phase', 'published_at', 'modified', 'emission_version', 'processing_software', 'gsd', 'sensitivity_mode', 'off_nadir', 'emission_auto', 'emission_uncertainty_auto', 'wind_speed_avg_auto', 'wind_speed_std_auto', 'wind_direction_avg_auto', 'wind_direction_std_auto', 'wind_source_auto', 'platform', 'provider', 'plume_tif', 'plume_png', 'con_tif', 'rgb_tif', 'rgb_png', 's2_path', 's2_pre_path(-90)', 's2_pre_pre_path (-360)', 's2_-7_path', 'std_ok', 'std_reason', 'bug', 's2_0_std_512', 's2_-7_std_512', 's2_-90_std_512', 's2_-360_std_512', 'has_m90', 'has_m360', 'date']
                    plume_id
0  GAO20191020t151740p0000-A
1  GAO20191020t153619p0000-A
2  GAO20191020t153619p0000-B
s2_0_std_512 True ['/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_-790360_512/GAO20191020t151740p0000-A/s2_0_std_512.tif', '/mnt/engg-leung/R